# 🏦 Bank Marketing - Term Deposit Subscription Prediction

**KD34403 Machine Learning for Data Science — Group 3 Project**

| NO | NAME | MATRIC NO. |
|---|---|---|
| 1. | GOO XIN YI | BI23110093 |
| 2. | TAM YI QING | BI23110092 |
| 3. | ONG CHIA YIN | BI23110241 |
| 4. | CHOK WENG HIN | BI23110182 |
| 5. | VOO XIN ROU | BI23110233 |

**Dataset:** UCI Bank Marketing (`bank-additional-full.csv`) — 41,188 rows, 20 features

**Core Question:** *Can we predict whether a bank client will subscribe to a term deposit based on their demographics and campaign interaction features?*
 

# MILESTONE 1 — DATA PIPELINE

## Step 0: Setup — Import Libraries

In [ ]:
# Data handling
import pandas as pd
import numpy as np

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

print('✅ libraries imported | Random seed:', RANDOM_STATE)

## Step 1: Problem Definition

| Aspect | Details |
|---|---|
| **Business Question** | Which clients are most likely to subscribe to a term deposit after a phone campaign? |
| **ML Problem Type** | Binary Classification |
| **Input (X)** | Client demographics, financial status, past campaign interactions, macroeconomic indicators |
| **Output (y)** | Subscribed = 1 (yes) / Not subscribed = 0 (no) |
| **Success Metrics** | Balanced Accuracy, ROC-AUC, F1-Score |
| **Stakeholders** | Bank marketing teams, campaign managers |

**Importance of this study:** Helps banks identify likely subscribers, reducing marketing costs and improving campaign efficiency.

**Key Consideration:** The dataset is heavily imbalanced (~11% positive), contains sentinel values and missing entries, and includes post-call information (duration) that must be removed to prevent leakage. Proper preprocessing is critical before any model training.

## Step 2: Data Collection & Loading

In [ ]:
import os, sys, urllib.request, zipfile, shutil

DATA_FILE = 'bank-additional-full.csv'
ZIP_URL   = 'https://archive.ics.uci.edu/static/public/222/bank+marketing.zip'

if not os.path.exists(DATA_FILE):
    print('Downloading Bank Marketing dataset from UCI...')
    urllib.request.urlretrieve(ZIP_URL, 'bank_marketing.zip')

    # Step 1: extract inner zip from outer zip
    with zipfile.ZipFile('bank_marketing.zip', 'r') as z:
        z.extract('bank-additional.zip', '.')
    os.remove('bank_marketing.zip')

    # Step 2: find and extract CSV — internal path may vary across UCI versions
    with zipfile.ZipFile('bank-additional.zip', 'r') as z:
        names = z.namelist()
        match = next((n for n in names if DATA_FILE in n), None)
        if match is None:
            raise FileNotFoundError(
                f'{DATA_FILE} not found in archive.\n'
                f'Archive contents: {names}'
            )
        z.extract(match, '.')
        # Move to working directory root if it was extracted into a subfolder
        if match != DATA_FILE:
            shutil.move(match, DATA_FILE)
            # Clean up any empty subfolder left behind
            folder = os.path.dirname(match)
            if folder and os.path.isdir(folder):
                shutil.rmtree(folder, ignore_errors=True)

    os.remove('bank-additional.zip')
    print(f'✅ {DATA_FILE} downloaded and ready')
else:
    print(f'✅ {DATA_FILE} already present — skipping download')

In [ ]:
# Load data & Version Check
df = pd.read_csv(DATA_FILE, sep=';')

# Confirm this is bank-additional-full and not the older bank-full (45,211 rows, 17 cols)
REQUIRED_COLS = ['euribor3m', 'emp.var.rate', 'nr.employed', 'cons.price.idx', 'cons.conf.idx']
missing_cols  = [c for c in REQUIRED_COLS if c not in df.columns]
if missing_cols:
    raise ValueError(f'Wrong dataset version — missing columns: {missing_cols}')

print('✅ Correct version: bank-additional-full.csv')

print('Sample rows:')
display(df.head())

In [ ]:
# info about the dataset
print('=' * 60)
print('Dataset Info:')
print('=' * 60)

print(f'Shape: {df.shape}')

print(f'\nData Types:')
print(df.dtypes.to_string())

# True NaN values
nan_counts = df.isnull().sum()
nan_counts = nan_counts[nan_counts > 0]
print(f'\nMissing Values (NaN):')
if nan_counts.empty:
    print('  None')
else:
    print(nan_counts.to_string())

print(f'\nCategorical Feature Unique Values:')
for col in df.select_dtypes(include='object').columns:
    vals = df[col].unique()[:5]
    print(f'  {col:15s}: {list(vals)} ...')

# Summary statistics
print('=' * 60)
print('Summary Statistics — Numeric Features:')
print('=' * 60)
display(df.describe().round(2))

# Key Observations
1. **Missing Values**
There are no true missing values (NaNs) in the dataset. However, several categorical features contain the value 'unknown', which represents missing or unrecorded information.

2. **Valid Special Category**
The value 'nonexistent' in the `poutcome` feature represents clients who were not previously contacted. Unlike 'unknown', this is a meaningful category and is retained.

3. **Macroeconomic Features**
The dataset includes several macroeconomic indicators:
- `emp.var.rate` – employment variation rate (quarterly)
- `cons.price.idx` – consumer price index
- `cons.conf.idx` – consumer confidence index
- `euribor3m` – 3-month Euro Interbank Offered Rate
- `nr.employed` – number of employees (in thousands)

These variables provide economic context that may influence customer behaviour.

4. **Sentinel Value in `pdays`**
The dataset documentation indicates that a special value (−1 in some versions) represents clients who were not previously contacted. In this dataset, −1 is not present (minimum value = 0), but the value 999 appears very frequently (as observed in summary statistics).

Given its dominance and lack of realistic interpretation as a number of days, 999 is inferred to function as a sentinel value indicating no previous contact.

## Step 3: Exploratory Data Analysis (EDA)

In [ ]:
# Class Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: raw counts
counts = df['y'].value_counts()
colors = ['#FF6B6B', '#4ECDC4']
bars = axes[0].bar(counts.index, counts.values, color=colors, edgecolor='white', width=0.5)
axes[0].set_title('Class Distribution: Term Deposit Subscription', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Clients')
axes[0].set_xlabel('Subscribed (y)')
for bar, count in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 200,
                 f'{count:,}\n({count/len(df)*100:.1f}%)', ha='center', fontweight='bold')

# Right: subscription rate by job
job_rate = df.groupby('job')['y'].apply(lambda x: (x=='yes').mean()).sort_values()
axes[1].barh(job_rate.index, job_rate.values, color='#4ECDC4', edgecolor='white')
axes[1].set_title('Subscription Rate by Job Type', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Subscription Rate')
axes[1].axvline(x=(df['y']=='yes').mean(), color='red', linestyle='--', label='Overall rate')
axes[1].legend()
plt.tight_layout()
plt.show()

# Imbalance metrics
pos_rate  = (df['y'] == 'yes').mean()
imb_ratio = counts['no'] / counts['yes']
print(f'Positive rate: {pos_rate:.1%}  |  Imbalance ratio: {imb_ratio:.1f}:1 (no:yes)')

In [ ]:
# Numeric Feature Distributions by Target

numeric_cols = ['age', 'campaign', 'previous', 'emp.var.rate',
                'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.ravel()

for idx, col in enumerate(numeric_cols):
    yes_vals = df[df['y'] == 'yes'][col]
    no_vals  = df[df['y'] == 'no'][col]
    axes[idx].hist(no_vals,  bins=30, alpha=0.6, color='#FF6B6B', label='No',  density=True, edgecolor='white')
    axes[idx].hist(yes_vals, bins=30, alpha=0.6, color='#4ECDC4', label='Yes', density=True, edgecolor='white')
    axes[idx].set_title(col, fontsize=11, fontweight='bold')
    axes[idx].legend(fontsize=8)
    axes[idx].grid(True, alpha=0.2)

fig.suptitle('Numeric Feature Distributions by Subscription Outcome', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Key observation: All five macroeconomic indicators show noticeably different distributions between customers who subscribed and those who did not. this suggests that the state of the economy at the time of the call plays a big role in whether someone will say yes')

In [ ]:
#  Correlation Heatmap (Numeric Features + Target)
df_temp = df.copy()
df_temp['y_bin'] = (df_temp['y'] == 'yes').astype(int)

corr_matrix = df_temp[numeric_cols + ['y_bin']].corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn',
            mask=mask, vmin=-1, vmax=1, center=0, square=True, linewidths=0.5)
plt.title('Correlation Heatmap — Numeric Features vs Target (y)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Correlation with target (y_bin), sorted by absolute value:')
print(corr_matrix['y_bin'].drop('y_bin').abs().sort_values(ascending=False).to_string())
print(" observation : nr.employed, euribor3m, and emp.var.rate are the strongest predictors "
      "but are highly intercorrelated (r up to 0.97), indicating multicollinearity.")

In [ ]:
# Categorical Feature Subscription Rates 
cat_cols_eda = ['marital', 'education', 'default', 'housing', 'loan',
                'contact', 'month', 'poutcome']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.ravel()
overall_rate = (df['y'] == 'yes').mean()

for idx, col in enumerate(cat_cols_eda):
    rate = df.groupby(col)['y'].apply(lambda x: (x=='yes').mean()).sort_values()
    rate.plot(kind='barh', ax=axes[idx], color='#4ECDC4', edgecolor='white')
    axes[idx].axvline(x=overall_rate, color='red', linestyle='--', alpha=0.7, label='Overall')
    axes[idx].set_title(f'Subscription Rate: {col}', fontsize=10, fontweight='bold')
    axes[idx].set_xlabel('Rate')
    axes[idx].legend(fontsize=7)
    axes[idx].grid(True, alpha=0.2)

plt.suptitle('Subscription Rate by Categorical Feature', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print("Key observation: Categorical features carry meaningful signal — "
      "poutcome, contact method, and month of contact are the strongest indicators of subscription likelihood.")

In [ ]:
# unknown Value Analysis 
unknown_counts = (df == 'unknown').sum()
unknown_counts = unknown_counts[unknown_counts > 0].sort_values()

plt.figure(figsize=(8, 5))
unknown_counts.plot(kind='barh', color='#FFA07A', edgecolor='white')
plt.title('Count of "unknown" Values per Column', fontsize=12, fontweight='bold')
plt.xlabel('Count')
plt.ylabel('Feature')
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

print('Columns with "unknown" values:')
print(unknown_counts.to_string())

print('It is noted that the default variable contains the largest number of "unknown" entries, which may limit its reliability as a predictor if left untreated.')

In [ ]:
#  pdays Sentinel Value Investigation
# pdays = 999 means the client was NOT previously contacted.
# Treating 999 as a numeric value would create a severely misleading feature.

print('pdays value distribution (top 10):')
print(df['pdays'].value_counts().head(10))

n_999 = (df['pdays'] == 999).sum()
print(f'\n999 sentinel: {n_999:,} rows ({n_999/len(df):.1%} of dataset)')

contacted     = df[df['pdays'] != 999]
not_contacted = df[df['pdays'] == 999]
print(f'\nSubscription rate — previously contacted:     {(contacted["y"]=="yes").mean():.1%}')
print(f'Subscription rate — NOT previously contacted: {(not_contacted["y"]=="yes").mean():.1%}')
print('\n→ Decision: replace pdays with binary flag `was_previously_contacted` and drop pdays.')
print('  This prevents the 999 sentinel from distorting any numeric model.')

In [ ]:
#  Outlier Detection (IQR method)
# Identifies outlier counts per numeric column using the 1.5×IQR rule.
# Informs whether to cap, transform, or leave outliers in preprocessing.

numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != 'y']  # exclude target

outlier_summary = []
for col in numeric_cols:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    n_out = ((df[col] < q1 - 1.5*iqr) | (df[col] > q3 + 1.5*iqr)).sum()
    outlier_summary.append({
        'Feature': col,
        'Q1': round(q1, 2),
        'Q3': round(q3, 2),
        'IQR': round(iqr, 2),
        'Outliers': n_out,
        'Outlier %': f'{n_out/len(df)*100:.1f}%'
    })

outlier_df = pd.DataFrame(outlier_summary).sort_values('Outliers', ascending=False)
print('Outlier Detection — IQR Method (1.5 × IQR rule)')
print('=' * 60)
display(outlier_df)

print('\nDecision: outliers are kept as-is.')
print('  - campaign outliers (many calls) reflect genuine client behaviour')
print('  - economic indicators have no true outliers — they are population-level indices')

## Step 4: Data Preprocessing 

### Preprocessing decisions 

| Decision | Justification |
|---|---|
| Drop `duration` |  This feature is only known after the call ends, so it leaks future information and cannot be used for real-world prediction.|
| Replace `pdays=999` with binary flag | The value 999 represents 'not previously contacted,' not a true numeric value. Converting it into a binary feature preserves its meaning without distorting the data. |
| Replace `'unknown'` with NaN | Treating 'unknown' as missing ensures consistent handling during imputation and avoids introducing misleading categories. |
| Ordinal encode `education` | Natural ordered hierarchy: illiterate to university.degree |
| One-hot encode other categoricals |These variables have no inherent order, so one-hot encoding prevents the model from assuming false relationships between categories. |
| Median impute numerics | Median is robust to extreme values and skewed distributions, making it suitable for real-world financial data.|
| Most-frequent impute categoricals | Replaces missing categories with the most common value, maintaining realistic distributions without introducing bias from arbitrary values. |
| Train-test split before preprocessing | Prevents data leakage by ensuring transformations are learned only from training data and then applied to unseen data.|
| StandardScaler on numerics | Ensures features are on the same scale, which is important for models like Logistic Regression and improves optimization stability.|


In [ ]:
# Initial Cleaning 
df_clean = df.copy()

# 1. Drop duration — data leakage
df_clean.drop(columns=['duration'], inplace=True)
print('✅ Dropped: duration (post-hoc leakage feature)')

# 2. Handle pdays sentinel — create binary flag, then drop pdays
df_clean['was_previously_contacted'] = (df_clean['pdays'] != 999).astype(int)
df_clean.drop(columns=['pdays'], inplace=True)
print('✅ Created: was_previously_contacted | Dropped: pdays')

# 3. Replace 'unknown' strings with NaN for proper imputation
unknown_counts = (df_clean == 'unknown').sum()
df_clean.replace('unknown', np.nan, inplace=True)
print(f'\n✅ Replaced "unknown" → NaN')
print('  Affected columns:')
print(unknown_counts[unknown_counts > 0].to_string())

# 4. Encode target
df_clean['y'] = (df_clean['y'] == 'yes').astype(int)
print(f'\n✅ Target encoded: yes=1, no=0')
print(f'  Distribution: {df_clean["y"].value_counts().to_dict()}')

In [ ]:
# Separate Features and Target
X = df_clean.drop(columns=['y'])
y = df_clean['y']

cat_cols_model = X.select_dtypes(include='object').columns.tolist()
num_cols_model = X.select_dtypes(include=np.number).columns.tolist()

print(f'Total features: {X.shape[1]}  ({len(num_cols_model)} numeric, {len(cat_cols_model)} categorical)')
print(f'\nNumeric ({len(num_cols_model)}):     {num_cols_model}')
print(f'Categorical ({len(cat_cols_model)}): {cat_cols_model}')

In [ ]:
# Stratified Train-Test Split
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print(f'Train: {X_train_raw.shape[0]:,} samples | Test: {X_test_raw.shape[0]:,} samples')
print(f'Train positive rate: {y_train_raw.mean():.3f}  |  Test positive rate: {y_test_raw.mean():.3f}')
print('✅ Stratified split preserves class ratio in both sets.')

In [ ]:
# Imputation (fit on train only)
# Numeric: median imputation (robust to outliers in age, campaign, balance)
# Categorical: most-frequent imputation (safe for job, education, marital)
num_imputer = SimpleImputer(strategy='median')
X_train_num = pd.DataFrame(
    num_imputer.fit_transform(X_train_raw[num_cols_model]),
    columns=num_cols_model, index=X_train_raw.index
)
X_test_num = pd.DataFrame(
    num_imputer.transform(X_test_raw[num_cols_model]),
    columns=num_cols_model, index=X_test_raw.index
)

cat_imputer = SimpleImputer(strategy='most_frequent')
X_train_cat_imp = pd.DataFrame(
    cat_imputer.fit_transform(X_train_raw[cat_cols_model]),
    columns=cat_cols_model, index=X_train_raw.index
)
X_test_cat_imp = pd.DataFrame(
    cat_imputer.transform(X_test_raw[cat_cols_model]),
    columns=cat_cols_model, index=X_test_raw.index
)

print('✅ Numeric: median imputation (fit on train, applied to test)')
print('✅ Categorical: most-frequent imputation (fit on train, applied to test)')

In [ ]:
# Encoding
# Education has a natural ordinal hierarchy → ordinal encoding.
# All other categoricals have no inherent order → one-hot encoding.

edu_order = ['illiterate', 'basic.4y', 'basic.6y', 'basic.9y',
             'high.school', 'professional.course', 'university.degree']
edu_map = {v: i for i, v in enumerate(edu_order)}  # illiterate=0, university=6

X_train_cat_enc = X_train_cat_imp.copy()
X_test_cat_enc  = X_test_cat_imp.copy()

# -1 for any remaining NaN after imputation (edge case: unseen value)
X_train_cat_enc['education'] = X_train_cat_enc['education'].map(edu_map).fillna(-1)
X_test_cat_enc['education']  = X_test_cat_enc['education'].map(edu_map).fillna(-1)

# One-hot encode the remaining categoricals
other_cat = [c for c in cat_cols_model if c != 'education']
X_train_ohe = pd.get_dummies(X_train_cat_enc, columns=other_cat, drop_first=True)
X_test_ohe  = pd.get_dummies(X_test_cat_enc,  columns=other_cat, drop_first=True)

# Align columns — test set may be missing dummy columns if a category was rare
X_test_ohe = X_test_ohe.reindex(columns=X_train_ohe.columns, fill_value=0)

print(f'✅ Ordinal: education ({len(edu_order)} levels, 0=illiterate → 6=university)')
print(f'✅ One-hot: {other_cat}')
print(f'   Categorical block expanded: → {X_train_ohe.shape[1]} columns')

In [ ]:
# Assemble Baseline Feature Matrix & Scale
# Reset indices to avoid mismatch
X_train_num = X_train_num.reset_index(drop=True)
X_test_num  = X_test_num.reset_index(drop=True)
X_train_ohe = X_train_ohe.reset_index(drop=True)
X_test_ohe  = X_test_ohe.reset_index(drop=True)
y_train_raw = y_train_raw.reset_index(drop=True)
y_test_raw  = y_test_raw.reset_index(drop=True)

# Identify continuous numeric columns (exclude binary flags)
binary_cols = ['was_previously_contacted']  
continuous_cols = [col for col in X_train_num.columns if col not in binary_cols]

# Initialize scaler
scaler = StandardScaler()

# Scale only continuous features
X_train_num_scaled = X_train_num.copy()
X_test_num_scaled  = X_test_num.copy()

X_train_num_scaled[continuous_cols] = scaler.fit_transform(X_train_num[continuous_cols])
X_test_num_scaled[continuous_cols]  = scaler.transform(X_test_num[continuous_cols])

# Combine scaled numeric features with one-hot encoded categorical features
X_train_scaled = pd.concat([X_train_num_scaled, X_train_ohe], axis=1)
X_test_scaled  = pd.concat([X_test_num_scaled, X_test_ohe], axis=1)

baseline_feature_names = X_train_scaled.columns.tolist()

print(f'✅ Baseline feature matrix assembled and scaled')
print(f'   Train: {X_train_scaled.shape}  |  Test: {X_test_scaled.shape}')
print(f'Sanity check — NaN in train: {X_train_scaled.isnull().sum().sum()}')
print(f'Sanity check — NaN in test:  {X_test_scaled.isnull().sum().sum()}')

print('\nExample scaled features (first 3 samples):')
display(X_train_scaled.head(3).round(2))

## Baseline Model: Logistic Regression
- only client demographics is used for training

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    f1_score, roc_auc_score,
    confusion_matrix, classification_report, roc_curve
)
import matplotlib.pyplot as plt
import seaborn as sns

# Explicitly select demographic-only features from scaled matrix
demo_cols_baseline = [
    'age', 'education',
    'job_blue-collar', 'job_entrepreneur', 'job_housemaid',
    'job_management', 'job_retired', 'job_self-employed',
    'job_services', 'job_student', 'job_technician', 'job_unemployed',
    'marital_married', 'marital_single',
    'default_yes', 'default_unknown',
    'housing_yes', 'housing_unknown',
    'loan_yes', 'loan_unknown'
]

X_train_baseline = X_train_scaled[[c for c in demo_cols_baseline if c in X_train_scaled.columns]]
X_test_baseline  = X_test_scaled[[c for c in demo_cols_baseline if c in X_test_scaled.columns]]

print(f'✅ Baseline feature set: {X_train_baseline.shape[1]} features (demographic only)')
print(f'   Features: {X_train_baseline.columns.tolist()}')

lr_baseline = LogisticRegression(
    solver='saga',
    penalty='l2',
    C=1.0,
    class_weight='balanced',
    max_iter=1000,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

print('\n✅ Baseline Logistic Regression architecture defined')
print(f'   Solver       : saga')
print(f'   Penalty      : L2')
print(f'   C            : 1.0  (default, no tuning)')
print(f'   class_weight : Balanced')

# Train
lr_baseline.fit(X_train_baseline, y_train_raw)
print('✅ Baseline model trained')

# Predict
y_pred_baseline       = lr_baseline.predict(X_test_baseline)
y_pred_proba_baseline = lr_baseline.predict_proba(X_test_baseline)[:, 1]

# Metrics
accuracy_baseline     = accuracy_score(y_test_raw, y_pred_baseline)
balanced_acc_baseline = balanced_accuracy_score(y_test_raw, y_pred_baseline)
f1_baseline           = f1_score(y_test_raw, y_pred_baseline)
roc_auc_baseline      = roc_auc_score(y_test_raw, y_pred_proba_baseline)

print('\nBASELINE MODEL PERFORMANCE')
print('=' * 60)
print(f'Accuracy:          {accuracy_baseline:.1%}')
print(f'Balanced Accuracy: {balanced_acc_baseline:.1%}')
print(f'F1-Score:          {f1_baseline:.1%}')
print(f'ROC-AUC:           {roc_auc_baseline:.3f}')

# Plots
cm_baseline = confusion_matrix(y_test_raw, y_pred_baseline)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm_baseline, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Pred No', 'Pred Yes'],
            yticklabels=['Actually No', 'Actually Yes'])
axes[0].set_title('Confusion Matrix (Baseline)', fontweight='bold')

fpr, tpr, _ = roc_curve(y_test_raw, y_pred_proba_baseline)
axes[1].plot(fpr, tpr, color='#4ECDC4', lw=3, label=f'Baseline (AUC={roc_auc_baseline:.3f})')
axes[1].plot([0, 1], [0, 1], '--', color='#FF6B6B', label='Random')
axes[1].set_title('ROC Curve', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

print(classification_report(y_test_raw, y_pred_baseline, target_names=['No (0)', 'Yes (1)']))

## Feature Engineering

**Three goals**:
1. Fix multicollinearity — PCA on 5 correlated economic cols
2. Two interaction terms — poutcome×previous, campaign×previous
3. Age binning — 4 groups replacing raw age

_*All fit objects (PCA, scaler) fitted on training data only._

In [ ]:
# First split: 70% train, 30% temp
X_train_raw, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    stratify=y,          # keep class distribution
    random_state=RANDOM_STATE
)

# Second split: split temp into validation and test (15% each)
X_val_raw, X_test_raw, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,      # half of 30% = 15%
    stratify=y_temp,
    random_state=RANDOM_STATE
)

print(f"Train size: {len(X_train_raw)} ({len(X_train_raw)/len(X):.1%})")
print(f"Val size:   {len(X_val_raw)} ({len(X_val_raw)/len(X):.1%})")
print(f"Test size:  {len(X_test_raw)} ({len(X_test_raw)/len(X):.1%})")

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

df_tr  = X_train_raw.copy()
df_val = X_val_raw.copy()
df_te  = X_test_raw.copy()

# Re-impute (fit on train only)
all_num = X_train_raw.select_dtypes(include=np.number).columns.tolist()
all_cat = X_train_raw.select_dtypes(include='object').columns.tolist()

num_imp_eng = SimpleImputer(strategy='median')
df_tr[all_num] = num_imp_eng.fit_transform(df_tr[all_num])
df_val[all_num] = num_imp_eng.transform(df_val[all_num])
df_te[all_num] = num_imp_eng.transform(df_te[all_num])

cat_imp_eng = SimpleImputer(strategy='most_frequent')
df_tr[all_cat] = cat_imp_eng.fit_transform(df_tr[all_cat])
df_val[all_cat] = cat_imp_eng.transform(df_val[all_cat])
df_te[all_cat] = cat_imp_eng.transform(df_te[all_cat])


In [ ]:
# =======================================
# Goal 1: Fix multicollinearity via PCA
# =======================================

# Group 1: labour-market block (r=0.97)
econ_block = ['emp.var.rate', 'euribor3m', 'nr.employed']

# Group 2: consumer sentiment block
cons_block  = ['cons.price.idx', 'cons.conf.idx']

# Scale each block before PCA (PCA is sensitive to scale)
scaler_econ = StandardScaler()
scaler_cons = StandardScaler()

econ_tr = scaler_econ.fit_transform(df_tr[econ_block])
econ_val = scaler_econ.transform(df_val[econ_block])
econ_te = scaler_econ.transform(df_te[econ_block])

cons_tr = scaler_cons.fit_transform(df_tr[cons_block])
cons_val = scaler_cons.transform(df_val[cons_block])
cons_te = scaler_cons.transform(df_te[cons_block])

# PCA → 1 component each (captures >95% variance given the high correlation)
pca_econ = PCA(n_components=1, random_state=RANDOM_STATE)
pca_cons = PCA(n_components=1, random_state=RANDOM_STATE)

df_tr['econ_index']  = pca_econ.fit_transform(econ_tr)
df_val['econ_index'] = pca_econ.transform(econ_val)
df_te['econ_index']  = pca_econ.transform(econ_te)

df_tr['cons_index']  = pca_cons.fit_transform(cons_tr)
df_val['cons_index'] = pca_cons.transform(cons_val)
df_te['cons_index']  = pca_cons.transform(cons_te)

print(f'econ_index explains {pca_econ.explained_variance_ratio_[0]:.1%} of variance in {econ_block}')
print(f'cons_index explains {pca_cons.explained_variance_ratio_[0]:.1%} of variance in {cons_block}')

In [ ]:
# =====================================
# Goal 2: Two interaction terms
# =====================================

# Interaction 1: previous campaign success × was previously contacted
# poutcome = 'success' is the strongest single categorical signal
df_tr['prev_success'] = (df_tr['poutcome'] == 'success').astype(int)
df_val['prev_success'] = (df_val['poutcome'] == 'success').astype(int)
df_te['prev_success'] = (df_te['poutcome'] == 'success').astype(int)

df_tr['poutcome_x_contact'] = df_tr['prev_success'] * df_tr['was_previously_contacted']
df_val['poutcome_x_contact'] = df_val['prev_success'] * df_val['was_previously_contacted']
df_te['poutcome_x_contact'] = df_te['prev_success'] * df_te['was_previously_contacted']

# Interaction 2: campaign effort × previous contacts
# High campaign calls + low previous = first-time aggressive outreach
# Low campaign calls + high previous = warm re-engagement
df_tr['campaign_x_previous'] = df_tr['campaign'] * (df_tr['previous'] + 1)
df_val['campaign_x_previous'] = df_val['campaign'] * (df_val['previous'] + 1)
df_te['campaign_x_previous'] = df_te['campaign'] * (df_te['previous'] + 1)

In [ ]:
# =====================================================
# Goal 3: Age binning (nonlinear subscription pattern)
# =====================================================
# young (<30): less financially stable, low subscription
# middle (30–45): busy with mortgages/family, moderate
# senior (45–60): peak savings period, higher subscription
# retired (60+): conservative investors, high subscription

bins   = [0, 30, 45, 60, 100]
labels = [0, 1, 2, 3]   # ordinal: young=0, retired=3

df_tr['age_group'] = pd.cut(df_tr['age'], bins=bins, labels=labels, right=False).astype(int)
df_val['age_group'] = pd.cut(df_val['age'], bins=bins, labels=labels, right=False).astype(int)
df_te['age_group'] = pd.cut(df_te['age'], bins=bins, labels=labels, right=False).astype(int)

In [ ]:
# Assemble enhanced feature matrix

# Numeric: baseline cols (drop raw age, replace with age_group)
#          + engineered numerics
#          + drop the 5 raw economic cols (replaced by PCA components)

eng_num_cols = [
    'age_group', 'campaign', 'previous', 'was_previously_contacted',
    'econ_index', 'cons_index',
    'prev_success', 'poutcome_x_contact', 'campaign_x_previous'
]

# Categorical: same as baseline (education ordinal, rest one-hot)
eng_cat_cols = [
    'job', 'marital', 'education',
    'default', 'housing', 'loan',
    'contact', 'month', 'day_of_week',
    'poutcome'
]

df_tr['education'] = df_tr['education'].map(edu_map).fillna(-1)
df_val['education'] = df_val['education'].map(edu_map).fillna(-1)
df_te['education'] = df_te['education'].map(edu_map).fillna(-1)

ohe_cols_eng = [c for c in eng_cat_cols if c != 'education']
df_val_ohe  = pd.get_dummies(df_val[ohe_cols_eng],  drop_first=True)
df_tr_ohe = pd.get_dummies(df_tr[ohe_cols_eng], drop_first=True)
df_te_ohe  = pd.get_dummies(df_te[ohe_cols_eng],  drop_first=True)

df_val_ohe = df_val_ohe.reindex(columns=df_tr_ohe.columns, fill_value=0)
df_te_ohe  = df_te_ohe.reindex(columns=df_tr_ohe.columns, fill_value=0)

X_eng_tr = pd.concat([
    df_tr[eng_num_cols + ['education']].reset_index(drop=True),
    df_tr_ohe.reset_index(drop=True)
], axis=1)

X_eng_val = pd.concat([
    df_val[eng_num_cols + ['education']].reset_index(drop=True),
    df_val_ohe.reset_index(drop=True)
], axis=1)

X_eng_te = pd.concat([
    df_te[eng_num_cols + ['education']].reset_index(drop=True),
    df_te_ohe.reset_index(drop=True)
], axis=1)

# Scale to fit on train only
binary_eng  = ['was_previously_contacted', 'prev_success', 'poutcome_x_contact']
scale_eng   = [c for c in eng_num_cols if c not in binary_eng]

scaler_eng = StandardScaler()
X_eng_tr[scale_eng] = scaler_eng.fit_transform(X_eng_tr[scale_eng])
X_eng_val[scale_eng] = scaler_eng.transform(X_eng_val[scale_eng])
X_eng_te[scale_eng] = scaler_eng.transform(X_eng_te[scale_eng])

enhanced_feature_names = X_eng_tr.columns.tolist()

print(f'\nBaseline features : {X_train_baseline.shape[1]}')
print(f'Enhanced features : {X_eng_tr.shape[1]}')
print(f'NaN in train: {X_eng_tr.isnull().sum().sum()} | NaN in validation: {X_eng_val.isnull().sum().sum()} | NaN in test: {X_eng_te.isnull().sum().sum()}')

# Logistic Regression Trained with Enhanced Feature and after Three-Way Stratified Split

In [ ]:
# ─────────────────────────────────────────────────────────────
# LOGISTIC REGRESSION — ENHANCED FEATURES
# ─────────────────────────────────────────────────────────────

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    f1_score, roc_auc_score,
    confusion_matrix, classification_report, roc_curve
)
import matplotlib.pyplot as plt
import seaborn as sns

lr_eng = LogisticRegression(
    solver       = 'saga',
    penalty      = 'l2',
    C            = 1.0,
    class_weight = 'balanced',
    max_iter     = 1000,
    random_state = RANDOM_STATE,
    n_jobs       = -1
)

# Train on the training split
lr_eng.fit(X_eng_tr, y_train)
print('✅ LR (enhanced features) trained on 60% training split')

# ── Predictions ───────────────────────────────────────────────
y_val_pred_lr   = lr_eng.predict(X_eng_val)
y_val_proba_lr  = lr_eng.predict_proba(X_eng_val)[:, 1]
y_test_pred_lr  = lr_eng.predict(X_eng_te)
y_test_proba_lr = lr_eng.predict_proba(X_eng_te)[:, 1]

# ── Metrics ───────────────────────────────────────────────────
def get_metrics(y_true, y_pred, y_proba):
    return {
        'Accuracy'         : accuracy_score(y_true, y_pred),
        'Balanced Accuracy': balanced_accuracy_score(y_true, y_pred),
        'F1-Score'         : f1_score(y_true, y_pred),
        'ROC-AUC'          : roc_auc_score(y_true, y_proba),
    }

val_m_lr  = get_metrics(y_val,  y_val_pred_lr,  y_val_proba_lr)
test_m_lr = get_metrics(y_test, y_test_pred_lr, y_test_proba_lr)

print('\nLR ENHANCED — VALIDATION vs TEST')
print('=' * 60)
print(f'{"Metric":<22} {"Validation":>12} {"Test":>12}')
print('-' * 48)
for k in val_m_lr:
    print(f'{k:<22} {val_m_lr[k]:>12.4f} {test_m_lr[k]:>12.4f}')

print(f'\nTest Classification Report:')
print(classification_report(y_test, y_test_pred_lr,
                            target_names=['No (0)', 'Yes (1)']))

# ── Plots ─────────────────────────────────────────────────────
cm_lr = confusion_matrix(y_test, y_test_pred_lr)
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_test_proba_lr)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Pred No', 'Pred Yes'],
            yticklabels=['Act No',  'Act Yes'])
axes[0].set_title('Confusion Matrix — LR Enhanced', fontweight='bold')

axes[1].plot(fpr_lr, tpr_lr, color='#4ECDC4', lw=2.5,
             label=f'LR Enhanced (AUC={test_m_lr["ROC-AUC"]:.3f})')
axes[1].plot([0, 1], [0, 1], '--', color='#FF6B6B', label='Random')
axes[1].set_title('ROC Curve — LR Enhanced', fontweight='bold')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# IMBALANCE HANDLING — THREE TECHNIQUES (LR)
# ─────────────────────────────────────────────────────────────

from imblearn.under_sampling import TomekLinks
from imblearn.over_sampling  import SMOTE
from imblearn.combine        import SMOTETomek

# ── Helper: train LR variant and return metrics ───────────────
def evaluate_lr(X_res, y_res, label, show_plots=True):
    model = LogisticRegression(
        solver       = 'saga',
        penalty      = 'l2',
        C            = 1.0,
        class_weight = 'balanced',
        max_iter     = 1000,
        random_state = RANDOM_STATE,
        n_jobs       = -1
    )
    model.fit(X_res, y_res)

    y_val_pred   = model.predict(X_eng_val)
    y_val_proba  = model.predict_proba(X_eng_val)[:, 1]
    y_test_pred  = model.predict(X_eng_te)
    y_test_proba = model.predict_proba(X_eng_te)[:, 1]

    val_m  = get_metrics(y_val,  y_val_pred,  y_val_proba)
    test_m = get_metrics(y_test, y_test_pred, y_test_proba)

    print(f'\n{"="*60}')
    print(f'  {label}')
    print(f'{"="*60}')
    print(f'{"Metric":<22} {"Validation":>12} {"Test":>12}')
    print(f'{"-"*48}')
    for k in val_m:
        print(f'{k:<22} {val_m[k]:>12.4f} {test_m[k]:>12.4f}')
    print(f'\nTest Classification Report:')
    print(classification_report(y_test, y_test_pred,
                                target_names=['No (0)', 'Yes (1)']))

    if show_plots:
        cm = confusion_matrix(y_test, y_test_pred)
        fpr, tpr, _ = roc_curve(y_test, y_test_proba)
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
                    xticklabels=['Pred No', 'Pred Yes'],
                    yticklabels=['Act No',  'Act Yes'])
        axes[0].set_title(f'Confusion Matrix — {label}', fontweight='bold')
        axes[1].plot(fpr, tpr, color='#4ECDC4', lw=2.5,
                     label=f'{label} (AUC={test_m["ROC-AUC"]:.3f})')
        axes[1].plot([0, 1], [0, 1], '--', color='#FF6B6B', label='Random')
        axes[1].set_title(f'ROC Curve — {label}', fontweight='bold')
        axes[1].set_xlabel('False Positive Rate')
        axes[1].set_ylabel('True Positive Rate')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

    return test_m, model


# ══════════════════════════════════════════════════════════════
# TECHNIQUE A — TOMEK LINKS (UNDERSAMPLING)
# ══════════════════════════════════════════════════════════════
print('\n' + '='*60)
print('  TECHNIQUE A — TOMEK LINKS (UNDERSAMPLING)')
print('='*60)

tl          = TomekLinks()
X_tl, y_tl = tl.fit_resample(X_eng_tr, y_train)
removed     = len(y_train) - len(y_tl)
print(f'Tomek Links removed {removed:,} borderline majority samples')
print(f'After resampling → {X_tl.shape[0]:,} samples | positive rate: {y_tl.mean():.3f}')

m_tl_lr, model_tl_lr = evaluate_lr(X_tl, y_tl, label='Technique A — TomekLinks')

# Before vs After
print('\nTECHNIQUE A — No Resample vs TomekLinks')
print(f'{"Metric":<22} {"No Resample":>13} {"TomekLinks":>12} {"Delta":>10}')
print('-'*60)
for k in test_m_lr:
    delta = m_tl_lr[k] - test_m_lr[k]
    arrow = '▲' if delta > 0 else '▼'
    print(f'{k:<22} {test_m_lr[k]:>13.4f} {m_tl_lr[k]:>12.4f} {arrow}{abs(delta):>8.4f}')


# ══════════════════════════════════════════════════════════════
# TECHNIQUE B — SMOTE (OVERSAMPLING)
# ══════════════════════════════════════════════════════════════
print('\n' + '='*60)
print('  TECHNIQUE B — SMOTE (OVERSAMPLING)')
print('='*60)

smote         = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
X_smt, y_smt = smote.fit_resample(X_eng_tr, y_train)
added         = len(y_smt) - len(y_train)
print(f'SMOTE synthesised {added:,} new minority samples')
print(f'After resampling → {X_smt.shape[0]:,} samples | positive rate: {y_smt.mean():.3f}')

m_smt_lr, model_smt_lr = evaluate_lr(X_smt, y_smt, label='Technique B — SMOTE')

# Before vs After
print('\nTECHNIQUE B — No Resample vs SMOTE')
print(f'{"Metric":<22} {"No Resample":>13} {"SMOTE":>12} {"Delta":>10}')
print('-'*60)
for k in test_m_lr:
    delta = m_smt_lr[k] - test_m_lr[k]
    arrow = '▲' if delta > 0 else '▼'
    print(f'{k:<22} {test_m_lr[k]:>13.4f} {m_smt_lr[k]:>12.4f} {arrow}{abs(delta):>8.4f}')


# ══════════════════════════════════════════════════════════════
# TECHNIQUE C — SMOTETomek (HYBRID)
# ══════════════════════════════════════════════════════════════
print('\n' + '='*60)
print('  TECHNIQUE C — SMOTETomek (HYBRID)')
print('='*60)

st_hybrid    = SMOTETomek(random_state=RANDOM_STATE)
X_st, y_st   = st_hybrid.fit_resample(X_eng_tr, y_train)
print(f'SMOTETomek → {X_st.shape[0]:,} samples | positive rate: {y_st.mean():.3f}')

m_st_lr, model_st_lr = evaluate_lr(X_st, y_st, label='Technique C — SMOTETomek')

# Before vs After
print('\nTECHNIQUE C — No Resample vs SMOTETomek')
print(f'{"Metric":<22} {"No Resample":>13} {"SMOTETomek":>12} {"Delta":>10}')
print('-'*60)
for k in test_m_lr:
    delta = m_st_lr[k] - test_m_lr[k]
    arrow = '▲' if delta > 0 else '▼'
    print(f'{k:<22} {test_m_lr[k]:>13.4f} {m_st_lr[k]:>12.4f} {arrow}{abs(delta):>8.4f}')

In [ ]:
# ─────────────────────────────────────────────────────────────
# IMBALANCE TECHNIQUE COMPARISON — LR
# ─────────────────────────────────────────────────────────────

lr_technique_results = {
    'No Resample' : test_m_lr,
    'TomekLinks'  : m_tl_lr,
    'SMOTE'       : m_smt_lr,
    'SMOTETomek'  : m_st_lr,
}

print('\n' + '='*70)
print('  IMBALANCE TECHNIQUE COMPARISON — LOGISTIC REGRESSION')
print('='*70)
print(f'{"Technique":<24} {"Accuracy":>10} {"Bal.Acc":>10} '
      f'{"F1-Score":>10} {"ROC-AUC":>10}')
print('-'*66)
for name, m in lr_technique_results.items():
    marker = ' ← baseline' if name == 'No Resample' else ''
    print(f'{name:<24} {m["Accuracy"]:>10.4f} {m["Balanced Accuracy"]:>10.4f} '
          f'{m["F1-Score"]:>10.4f} {m["ROC-AUC"]:>10.4f}{marker}')

# ── Bar chart — Balanced Accuracy & F1-Score ──────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
names  = list(lr_technique_results.keys())
colors = ['#AAAAAA', '#FF6B6B', '#4ECDC4', '#FFD93D']

for ax, metric in zip(axes, ['Balanced Accuracy', 'F1-Score']):
    vals = [lr_technique_results[n][metric] for n in names]
    bars = ax.bar(names, vals, color=colors, edgecolor='white', width=0.5)
    ax.set_title(metric, fontweight='bold')
    ax.set_ylabel(metric)
    ax.set_ylim(min(vals) - 0.05, max(vals) + 0.05)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.002,
                f'{val:.4f}', ha='center', fontsize=9, fontweight='bold')
    ax.grid(True, alpha=0.2)
    ax.tick_params(axis='x', rotation=15)

fig.suptitle('Imbalance Technique Comparison — Balanced Accuracy & F1-Score (LR)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# ── ROC overlay — all four ────────────────────────────────────
plt.figure(figsize=(9, 6))
line_colors = ['#AAAAAA', '#FF6B6B', '#4ECDC4', '#FFD93D']

for (name, model), col in zip(
    [('No Resample', lr_eng),
     ('TomekLinks',  model_tl_lr),
     ('SMOTE',       model_smt_lr),
     ('SMOTETomek',  model_st_lr)],
    line_colors
):
    proba       = model.predict_proba(X_te_lr)[:, 1]
    fpr, tpr, _ = roc_curve(y_test_lr, proba)
    auc         = roc_auc_score(y_test_lr, proba)
    plt.plot(fpr, tpr, lw=2, color=col, label=f'{name} (AUC={auc:.3f})')

plt.plot([0, 1], [0, 1], '--', color='grey', label='Random')
plt.title('ROC Overlay — No Resample vs Imbalance Techniques (LR)', fontweight='bold')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ── Auto-select best technique ────────────────────────────────
resample_only_lr = {k: v for k, v in lr_technique_results.items()
                    if k != 'No Resample'}

best_technique_lr = max(
    resample_only_lr,
    key=lambda k: (resample_only_lr[k]['Balanced Accuracy'],
                   resample_only_lr[k]['F1-Score'])
)

print(f'\n{"="*60}')
print(f'  ✅ BEST IMBALANCE TECHNIQUE (LR): {best_technique_lr}')
print(f'     Balanced Accuracy : {lr_technique_results[best_technique_lr]["Balanced Accuracy"]:.4f}')
print(f'     F1-Score          : {lr_technique_results[best_technique_lr]["F1-Score"]:.4f}')
print(f'     ROC-AUC           : {lr_technique_results[best_technique_lr]["ROC-AUC"]:.4f}')
print(f'{"="*60}')

In [ ]:
# ─────────────────────────────────────────────────────────────
# HYPERPARAMETER TUNING — LOGISTIC REGRESSION
# ─────────────────────────────────────────────────────────────

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
import numpy as np

# ── Select resampled data from best technique ─────────────────
lr_resample_data = {
    'TomekLinks' : (X_tl,  y_tl),
    'SMOTE'      : (X_smt, y_smt),
    'SMOTETomek' : (X_st,  y_st),
}

X_best_lr, y_best_lr = lr_resample_data[best_technique_lr]

# ══════════════════════════════════════════════════════════════
# ▼▼▼  USER-CONFIGURABLE HYPERPARAMETER SEARCH SETTINGS  ▼▼▼
# ══════════════════════════════════════════════════════════════
param_dist_lr = {
    'C'           : [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
    'penalty'     : ['l1', 'l2'],
    'solver'      : ['saga'],
    'class_weight': ['balanced'],
    'max_iter'    : [1000],
}
# ▲▲▲  END OF USER-CONFIGURABLE SECTION  ▲▲▲
# ══════════════════════════════════════════════════════════════

n_iter   = 20
cv_folds = 3

print('\n' + '='*60)
print(f'  HYPERPARAMETER TUNING — LR ({best_technique_lr})')
print('='*60)
print(f'  Search space summary:')
for param, values in param_dist_lr.items():
    print(f'   {param:<22}: {values}')
print(f'\n  n_iter   = {n_iter}  (random combinations to try)')
print(f'  cv_folds = {cv_folds}  (stratified cross-validation folds)')
print(f'  scoring  = roc_auc')

cv_strategy_lr = StratifiedKFold(n_splits=cv_folds, shuffle=True,
                                 random_state=RANDOM_STATE)

rscv_lr = RandomizedSearchCV(
    estimator           = LogisticRegression(random_state=RANDOM_STATE,
                                             n_jobs=-1),
    param_distributions = param_dist_lr,
    n_iter              = n_iter,
    scoring             = 'roc_auc',
    cv                  = cv_strategy_lr,
    verbose             = 1,
    random_state        = RANDOM_STATE,
    n_jobs              = -1,
    refit               = True
)

rscv_lr.fit(X_best_lr, y_best_lr)

print(f'\n✅ Best parameters found:')
for k, v in rscv_lr.best_params_.items():
    print(f'   {k:<22}: {v}')
print(f'\n   Best CV ROC-AUC: {rscv_lr.best_score_:.4f}')

lr_tuned = rscv_lr.best_estimator_

# ── Predictions ───────────────────────────────────────────────
y_tu_val_pred_lr   = lr_tuned.predict(X_eng_val)
y_tu_val_proba_lr  = lr_tuned.predict_proba(X_eng_val)[:, 1]
y_tu_test_pred_lr  = lr_tuned.predict(X_eng_te)
y_tu_test_proba_lr = lr_tuned.predict_proba(X_eng_te)[:, 1]

# ── Metrics ───────────────────────────────────────────────────
tuned_val_metrics_lr = get_metrics(y_val,  y_tu_val_pred_lr,  y_tu_val_proba_lr)
tuned_test_metrics_lr = get_metrics(y_test, y_tu_test_pred_lr, y_tu_test_proba_lr)

print(f'\nTUNED LR — {best_technique_lr}')
print('=' * 60)
print(f'{"Metric":<22} {"Validation":>12} {"Test":>12}')
print('-'*48)
for k in tuned_test_metrics_lr:
    print(f'{k:<22} {tuned_val_metrics_lr[k]:>12.4f} {tuned_test_metrics_lr[k]:>12.4f}')

print(f'\nTest Classification Report:')
print(classification_report(y_test, y_tu_test_pred_lr,
                            target_names=['No (0)', 'Yes (1)']))

# ── Plots ─────────────────────────────────────────────────────
cm_tu_lr        = confusion_matrix(y_test, y_tu_test_pred_lr)
fpr_tu_lr, tpr_tu_lr, _ = roc_curve(y_test, y_tu_test_proba_lr)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cm_tu_lr, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Pred No', 'Pred Yes'],
            yticklabels=['Act No',  'Act Yes'])
axes[0].set_title(f'Confusion Matrix — Tuned LR ({best_technique_lr})', fontweight='bold')

axes[1].plot(fpr_tu_lr, tpr_tu_lr, color='#4ECDC4', lw=2.5,
             label=f'Tuned LR (AUC={tuned_test_metrics_lr["ROC-AUC"]:.3f})')
axes[1].plot([0, 1], [0, 1], '--', color='grey', label='Random')
axes[1].set_title(f'ROC Curve — Tuned LR ({best_technique_lr})', fontweight='bold')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ── Coefficient plot (top 20 by absolute weight) ──────────────
coef_series = pd.Series(
    lr_tuned.coef_[0],
    index=X_eng_tr.columns
).reindex(pd.Series(lr_tuned.coef_[0],
                    index=X_eng_tr.columns).abs().sort_values(ascending=False).index)

plt.figure(figsize=(10, 6))
colors_coef = ['#4ECDC4' if v > 0 else '#FF6B6B' for v in coef_series.head(20)]
coef_series.head(20).plot(kind='bar', color=colors_coef, edgecolor='white')
plt.title('Top 20 Feature Coefficients — Tuned LR', fontweight='bold')
plt.ylabel('Coefficient Value')
plt.axhline(y=0, color='grey', linestyle='--', alpha=0.5)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# ── No Resample vs Tuned comparison ───────────────────────────
print('\n' + '='*60)
print(f'  NO RESAMPLE vs TUNED LR ({best_technique_lr})')
print('='*60)
print(f'{"Metric":<22} {"No Resample":>13} {"Tuned LR":>12} {"Delta":>10}')
print('-'*60)
for k in test_m_lr:
    delta = tuned_test_metrics_lr[k] - test_m_lr[k]
    arrow = '▲' if delta > 0 else ('▼' if delta < 0 else '–')
    print(f'{k:<22} {test_m_lr[k]:>13.4f} {tuned_test_metrics_lr[k]:>12.4f} '
          f'{arrow}{abs(delta):>8.4f}')

# ── ROC overlay — No Resample vs Tuned ───────────────────────
fpr_nr, tpr_nr, _ = roc_curve(y_test, lr_eng.predict_proba(X_eng_te)[:, 1])

plt.figure(figsize=(8, 6))
plt.plot(fpr_nr,    tpr_nr,    color='#FF6B6B', lw=2,
         label=f'No Resample (AUC={test_m_lr["ROC-AUC"]:.3f})')
plt.plot(fpr_tu_lr, tpr_tu_lr, color='#4ECDC4', lw=2,
         label=f'Tuned LR    (AUC={tuned_test_metrics_lr["ROC-AUC"]:.3f})')
plt.plot([0, 1], [0, 1], '--', color='grey', label='Random')
plt.title(f'ROC — No Resample vs Tuned LR ({best_technique_lr})', fontweight='bold')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('\n' + '='*60)
print('  FINAL SUMMARY — LOGISTIC REGRESSION')
print('='*60)
print(f'  Best imbalance technique : {best_technique_lr}')
print(f'  Best hyperparameters     : C={rscv_lr.best_params_["C"]}, '
      f'penalty={rscv_lr.best_params_["penalty"]}')
print(f'  No Resample — Bal.Acc : {test_m_lr["Balanced Accuracy"]:.4f} | '
      f'F1: {test_m_lr["F1-Score"]:.4f} | AUC: {test_m_lr["ROC-AUC"]:.4f}')
print(f'  Tuned LR    — Bal.Acc : {tuned_test_metrics_lr["Balanced Accuracy"]:.4f} | '
      f'F1: {tuned_test_metrics_lr["F1-Score"]:.4f} | AUC: {tuned_test_metrics_lr["ROC-AUC"]:.4f}')
improvement_lr = tuned_test_metrics_lr["ROC-AUC"] - test_m_lr["ROC-AUC"]
direction_lr   = "improvement" if improvement_lr >= 0 else "degradation"
print(f'  Tuning {direction_lr}      : {improvement_lr:+.4f} AUC')
print('='*60)

In [ ]:
# ─────────────────────────────────────────────────────────────
# BASELINE LR vs TUNED LR (TOMEKLINKS) — FINAL COMPARISON
# ─────────────────────────────────────────────────────────────

# ── Retrain tuned LR on TomekLinks data ───────────────────────
lr_final = LogisticRegression(
    C            = rscv_lr.best_params_['C'],
    penalty      = rscv_lr.best_params_['penalty'],
    solver       = 'saga',
    class_weight = 'balanced',
    max_iter     = 1000,
    random_state = RANDOM_STATE,
    n_jobs       = -1
)

lr_final.fit(X_tl, y_tl)
print(f'✅ Final LR trained on TomekLinks data')
print(f'   C={rscv_lr.best_params_["C"]} | penalty={rscv_lr.best_params_["penalty"]}')

# ── Predictions ───────────────────────────────────────────────
y_final_pred  = lr_final.predict(X_eng_te)
y_final_proba = lr_final.predict_proba(X_eng_te)[:, 1]

final_metrics = get_metrics(y_test, y_final_pred, y_final_proba)

# ── Head-to-head table ────────────────────────────────────────
# Baseline LR  = lr_baseline trained on demographic features only (Step 2)
# Tuned LR     = lr_final trained on enhanced features + TomekLinks + best C/penalty

print('\n' + '='*65)
print('  BASELINE LR vs TUNED LR — HEAD-TO-HEAD COMPARISON')
print('='*65)
print(f'{"Metric":<22} {"Baseline LR":>14} {"Tuned LR":>12} {"Delta":>10}')
print('-'*62)

baseline_ref = {
    'Accuracy'         : accuracy_baseline,
    'Balanced Accuracy': balanced_acc_baseline,
    'F1-Score'         : f1_baseline,
    'ROC-AUC'          : roc_auc_baseline,
}

for k in baseline_ref:
    delta = final_metrics[k] - baseline_ref[k]
    arrow = '▲' if delta > 0 else ('▼' if delta < 0 else '–')
    print(f'{k:<22} {baseline_ref[k]:>14.4f} {final_metrics[k]:>12.4f} '
          f'{arrow}{abs(delta):>8.4f}')

# ── Classification reports side by side ───────────────────────
print('\nBASELINE LR — Classification Report:')
print(classification_report(y_test_raw, y_pred_baseline,
                            target_names=['No (0)', 'Yes (1)']))

print('TUNED LR — Classification Report:')
print(classification_report(y_test, y_final_pred,
                            target_names=['No (0)', 'Yes (1)']))

# ── Confusion matrices side by side ───────────────────────────
cm_baseline_ref = confusion_matrix(y_test_raw, y_pred_baseline)
cm_final        = confusion_matrix(y_test, y_final_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, cm, label in [
    (axes[0], cm_baseline_ref, 'Baseline LR\n(Demographics only, No Resample)'),
    (axes[1], cm_final,        f'Tuned LR\n(Enhanced + TomekLinks, C={rscv_lr.best_params_["C"]})')
]:
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Pred No', 'Pred Yes'],
                yticklabels=['Act No',  'Act Yes'])
    ax.set_title(label, fontweight='bold')

fig.suptitle('Confusion Matrix — Baseline LR vs Tuned LR',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# ── ROC overlay ───────────────────────────────────────────────
fpr_base, tpr_base, _ = roc_curve(y_test_raw,    y_pred_proba_baseline)
fpr_final, tpr_final, _ = roc_curve(y_test, y_final_proba)

plt.figure(figsize=(8, 6))
plt.plot(fpr_base,  tpr_base,  color='#FF6B6B', lw=2,
         label=f'Baseline LR  (AUC={roc_auc_baseline:.3f})')
plt.plot(fpr_final, tpr_final, color='#4ECDC4', lw=2,
         label=f'Tuned LR     (AUC={final_metrics["ROC-AUC"]:.3f})')
plt.plot([0, 1], [0, 1], '--', color='grey', label='Random')
plt.title('ROC Curve — Baseline LR vs Tuned LR', fontweight='bold')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ── Bar chart — all four metrics ──────────────────────────────
metrics_list = ['Accuracy', 'Balanced Accuracy', 'F1-Score', 'ROC-AUC']
baseline_vals = [baseline_ref[m] for m in metrics_list]
tuned_vals    = [final_metrics[m] for m in metrics_list]

x     = np.arange(len(metrics_list))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, baseline_vals, width, label='Baseline LR',
               color='#FF6B6B', edgecolor='white')
bars2 = ax.bar(x + width/2, tuned_vals,    width, label='Tuned LR',
               color='#4ECDC4', edgecolor='white')

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.4f}', ha='center', fontsize=8, fontweight='bold')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.4f}', ha='center', fontsize=8, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(metrics_list)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.set_title('Baseline LR vs Tuned LR — All Metrics', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

# ── Final summary ─────────────────────────────────────────────
print('\n' + '='*65)
print('  FINAL SUMMARY — BASELINE LR vs TUNED LR')
print('='*65)
print(f'  Baseline LR  : Demographics only | No resampling | Default C=1.0, L2')
print(f'  Tuned LR     : Enhanced features | TomekLinks | '
      f'C={rscv_lr.best_params_["C"]}, {rscv_lr.best_params_["penalty"].upper()}')
print(f'\n  {"Metric":<22} {"Baseline":>10} {"Tuned":>10} {"Gain":>10}')
print(f'  {"-"*54}')
for k in baseline_ref:
    gain  = final_metrics[k] - baseline_ref[k]
    arrow = '▲' if gain > 0 else ('▼' if gain < 0 else '–')
    print(f'  {k:<22} {baseline_ref[k]:>10.4f} {final_metrics[k]:>10.4f} '
          f'{arrow}{abs(gain):>8.4f}')
print('='*65)

# Random Forest

In [ ]:
# Random Forest
# ─────────────────────────────────────────────────────────────
# STEP 0 — IMPORT LIBRARIES
# ─────────────────────────────────────────────────────────────
 
import warnings
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
 
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.ensemble        import RandomForestClassifier
from sklearn.metrics         import (
    accuracy_score, balanced_accuracy_score,
    f1_score, roc_auc_score,
    confusion_matrix, classification_report, roc_curve
)
 
from imblearn.under_sampling import TomekLinks
from imblearn.over_sampling  import SMOTE
from imblearn.combine        import SMOTETomek
 
warnings.filterwarnings('ignore')
 
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
 
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi']     = 100
 
print('✅ All libraries imported | RANDOM_STATE =', RANDOM_STATE)

In [ ]:
# ─────────────────────────────────────────────────────────────
# STEP 2 — THREE-WAY STRATIFIED SPLIT
# ─────────────────────────────────────────────────────────────
# The LR preprocessing block already produced a stratified
# train/test split (80/20).  We now carve a validation set
# out of the existing training portion so that:
#
#   X_eng_tr (80%)  ──► 75% TRAIN  +  25% VAL
#                        = 60% of full data  +  20% of full data
#
# Final proportions: ~60% Train | ~20% Val | 20% Test
#
# All splits are stratified to preserve the ~11% positive rate.
# The test set (X_eng_te / y_test) remains untouched — it was
# locked away by the LR preprocessing and is never resampled.
 
X_tr_eng, X_vl_eng, y_train_rf, y_val = train_test_split(
    X_eng_tr, y_train,
    test_size    = 0.25,
    stratify     = y_train,
    random_state = RANDOM_STATE
)
 
# Alias test set from LR preprocessing for consistency
X_te_eng = X_eng_te.copy()
y_test_rf = y_test.copy()
 
# Reset indices across all splits
X_tr_eng   = X_tr_eng.reset_index(drop=True)
X_vl_eng   = X_vl_eng.reset_index(drop=True)
X_te_eng   = X_te_eng.reset_index(drop=True)
y_train_rf = y_train_rf.reset_index(drop=True)
y_val      = y_val.reset_index(drop=True)
y_test_rf  = y_test_rf.reset_index(drop=True)
 
print('THREE-WAY STRATIFIED SPLIT')
print('=' * 50)
print(f'Train : {X_tr_eng.shape[0]:>6,} samples | positive rate: {y_train_rf.mean():.3f}')
print(f'Val   : {X_vl_eng.shape[0]:>6,} samples | positive rate: {y_val.mean():.3f}')
print(f'Test  : {X_te_eng.shape[0]:>6,} samples | positive rate: {y_test_rf.mean():.3f}')
print('✅ Stratified split preserves class ratio across all three sets.')

In [ ]:
# ─────────────────────────────────────────────────────────────
# STEP 4 — BASELINE RANDOM FOREST
#           (no resampling, no hyperparameter tuning)
# ─────────────────────────────────────────────────────────────
# We first establish a baseline using the raw (unbalanced)
# training set with default hyperparameters.
# class_weight='balanced' is used to give the model a soft
# awareness of imbalance without any resampling.
# This result becomes the reference point for comparing the
# three imbalance techniques in Step 5.
 
# ── Helper: train one RF variant and return metrics + model ───
def evaluate_rf(X_res, y_res, params, label, show_plots=True):
    """
    Trains a RandomForestClassifier on (X_res, y_res).
    Evaluates on the fixed val and test sets.
    Returns (test_metrics_dict, fitted_model).
    """
    model = RandomForestClassifier(**params)
    model.fit(X_res, y_res)
 
    y_val_pred   = model.predict(X_vl_eng)
    y_val_proba  = model.predict_proba(X_vl_eng)[:, 1]
    y_test_pred  = model.predict(X_te_eng)
    y_test_proba = model.predict_proba(X_te_eng)[:, 1]
 
    def get_metrics(y_true, y_pred, y_proba):
        return {
            'Accuracy'         : accuracy_score(y_true, y_pred),
            'Balanced Accuracy': balanced_accuracy_score(y_true, y_pred),
            'F1-Score'         : f1_score(y_true, y_pred),
            'ROC-AUC'          : roc_auc_score(y_true, y_proba),
        }
 
    val_m  = get_metrics(y_val,      y_val_pred,  y_val_proba)
    test_m = get_metrics(y_test_rf,  y_test_pred, y_test_proba)
 
    print(f'\n{"="*60}')
    print(f'  {label}')
    print(f'{"="*60}')
    print(f'{"Metric":<22} {"Validation":>12} {"Test":>12}')
    print(f'{"-"*48}')
    for k in val_m:
        print(f'{k:<22} {val_m[k]:>12.4f} {test_m[k]:>12.4f}')
    print(f'\nTest Classification Report:')
    print(classification_report(y_test_rf, y_test_pred,
                                target_names=['No (0)', 'Yes (1)']))
 
    if show_plots:
        cm = confusion_matrix(y_test_rf, y_test_pred)
        fpr, tpr, _ = roc_curve(y_test_rf, y_test_proba)
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
                    xticklabels=['Pred No', 'Pred Yes'],
                    yticklabels=['Act No',  'Act Yes'])
        axes[0].set_title(f'Confusion Matrix — {label}', fontweight='bold')
        axes[1].plot(fpr, tpr, color='#4ECDC4', lw=2.5,
                     label=f'{label} (AUC={test_m["ROC-AUC"]:.3f})')
        axes[1].plot([0,1],[0,1],'--', color='#FF6B6B', label='Random')
        axes[1].set_title(f'ROC Curve — {label}', fontweight='bold')
        axes[1].set_xlabel('False Positive Rate')
        axes[1].set_ylabel('True Positive Rate')
        axes[1].legend(); axes[1].grid(True, alpha=0.3)
        plt.tight_layout(); plt.show()
 
    return test_m, model
 
# ── Default RF params (reused across Steps 4 and 5) ──────────
DEFAULT_PARAMS = dict(
    n_estimators = 100,
    max_depth    = None,
    class_weight = 'balanced',
    random_state = RANDOM_STATE,
    n_jobs       = -1
)
 
print('\n' + '='*60)
print('  STEP 4 — BASELINE RF (no resampling, no tuning)')
print('='*60)
 
m_no_resample, model_nores = evaluate_rf(
    X_tr_eng, y_train_rf, DEFAULT_PARAMS,
    label='Baseline RF (no resampling)'
)

In [ ]:
# ─────────────────────────────────────────────────────────────
# STEP 5 — IMBALANCE HANDLING (compare 3 techniques)
# ─────────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════
# TECHNIQUE A — TOMEK LINKS (UNDERSAMPLING)
# ══════════════════════════════════════════════════════════════
print('\n' + '='*60)
print('  TECHNIQUE A — TOMEK LINKS (UNDERSAMPLING)')
print('='*60)
 
tl          = TomekLinks()
X_tl, y_tl = tl.fit_resample(X_tr_eng, y_train_rf)
removed     = len(y_train_rf) - len(y_tl)
print(f'Tomek Links removed {removed:,} borderline majority samples')
print(f'After resampling → {X_tl.shape[0]:,} samples | positive rate: {y_tl.mean():.3f}')
 
m_tl, model_tl = evaluate_rf(
    X_tl, y_tl, DEFAULT_PARAMS,
    label='Technique A — TomekLinks'
)
 
# Before vs After — Technique A
print('\nTECHNIQUE A — Baseline vs TomekLinks')
print(f'{"Metric":<22} {"Baseline":>13} {"TomekLinks":>12} {"Delta":>10}')
print('-'*60)
for k in m_no_resample:
    delta = m_tl[k] - m_no_resample[k]
    arrow = '▲' if delta > 0 else ('▼' if delta < 0 else '–')
    print(f'{k:<22} {m_no_resample[k]:>13.4f} {m_tl[k]:>12.4f} '
          f'{arrow}{abs(delta):>8.4f}')

In [ ]:
# ══════════════════════════════════════════════════════════════
# TECHNIQUE B — SMOTE (OVERSAMPLING)
# ══════════════════════════════════════════════════════════════
print('\n' + '='*60)
print('  TECHNIQUE B — SMOTE (OVERSAMPLING)')
print('='*60)
 
smote         = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
X_smt, y_smt = smote.fit_resample(X_tr_eng, y_train_rf)
added         = len(y_smt) - len(y_train_rf)
print(f'SMOTE synthesised {added:,} new minority samples')
print(f'After resampling → {X_smt.shape[0]:,} samples | positive rate: {y_smt.mean():.3f}')
 
m_smt, model_smt = evaluate_rf(
    X_smt, y_smt, DEFAULT_PARAMS,
    label='Technique B — SMOTE'
)
 
# Before vs After — Technique B
print('\nTECHNIQUE B — Baseline vs SMOTE')
print(f'{"Metric":<22} {"Baseline":>13} {"SMOTE":>12} {"Delta":>10}')
print('-'*60)
for k in m_no_resample:
    delta = m_smt[k] - m_no_resample[k]
    arrow = '▲' if delta > 0 else ('▼' if delta < 0 else '–')
    print(f'{k:<22} {m_no_resample[k]:>13.4f} {m_smt[k]:>12.4f} '
          f'{arrow}{abs(delta):>8.4f}')

In [ ]:
# ══════════════════════════════════════════════════════════════
# TECHNIQUE C — SMOTETomek (HYBRID)
# ══════════════════════════════════════════════════════════════
print('\n' + '='*60)
print('  TECHNIQUE C — SMOTETomek (HYBRID)')
print('='*60)
 
st_hybrid    = SMOTETomek(random_state=RANDOM_STATE)
X_st, y_st   = st_hybrid.fit_resample(X_tr_eng, y_train_rf)
print(f'SMOTETomek → {X_st.shape[0]:,} samples | positive rate: {y_st.mean():.3f}')
 
m_st, model_st = evaluate_rf(
    X_st, y_st, DEFAULT_PARAMS,
    label='Technique C — SMOTETomek'
)
 
# Before vs After — Technique C
print('\nTECHNIQUE C — Baseline vs SMOTETomek')
print(f'{"Metric":<22} {"Baseline":>13} {"SMOTETomek":>12} {"Delta":>10}')
print('-'*60)
for k in m_no_resample:
    delta = m_st[k] - m_no_resample[k]
    arrow = '▲' if delta > 0 else ('▼' if delta < 0 else '–')
    print(f'{k:<22} {m_no_resample[k]:>13.4f} {m_st[k]:>12.4f} '
          f'{arrow}{abs(delta):>8.4f}')

In [ ]:
# ─────────────────────────────────────────────────────────────
# STEP 6 — CHOOSE BEST IMBALANCE TECHNIQUE
# ─────────────────────────────────────────────────────────────
# We rank the three techniques by Balanced Accuracy (primary)
# and F1-Score (tiebreaker).
# These two metrics are prioritised over raw Accuracy and
# ROC-AUC because the dataset is heavily imbalanced (~9:1).
# Raw Accuracy is dominated by the majority class and can be
# misleading. Balanced Accuracy gives equal weight to both
# classes. F1-Score captures the precision-recall trade-off
# which is critical for correctly identifying subscribers.
 
technique_results = {
    'Baseline (no resample)': m_no_resample,
    'TomekLinks'            : m_tl,
    'SMOTE'                 : m_smt,
    'SMOTETomek'            : m_st,
}
 
resample_data = {
    'TomekLinks': (X_tl,  y_tl),
    'SMOTE'     : (X_smt, y_smt),
    'SMOTETomek': (X_st,  y_st),
}
 
print('\n' + '='*70)
print('  STEP 6 — IMBALANCE TECHNIQUE COMPARISON (same Default RF)')
print('='*70)
print(f'{"Technique":<24} {"Accuracy":>10} {"Bal.Acc":>10} '
      f'{"F1-Score":>10} {"ROC-AUC":>10}')
print('-'*66)
for name, m in technique_results.items():
    marker = ' ← baseline' if name == 'Baseline (no resample)' else ''
    print(f'{name:<24} {m["Accuracy"]:>10.4f} {m["Balanced Accuracy"]:>10.4f} '
          f'{m["F1-Score"]:>10.4f} {m["ROC-AUC"]:>10.4f}{marker}')
 
# Visual comparison — bar chart (Balanced Accuracy & F1-Score)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
names   = list(technique_results.keys())
colors  = ['#AAAAAA', '#FF6B6B', '#4ECDC4', '#FFD93D']
 
for ax, metric in zip(axes, ['Balanced Accuracy', 'F1-Score']):
    vals = [technique_results[n][metric] for n in names]
    bars = ax.bar(names, vals, color=colors, edgecolor='white', width=0.5)
    ax.set_title(metric, fontweight='bold')
    ax.set_ylabel(metric)
    ax.set_ylim(min(vals) - 0.05, max(vals) + 0.05)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.002,
                f'{val:.4f}', ha='center', fontsize=9, fontweight='bold')
    ax.grid(True, alpha=0.2)
    ax.tick_params(axis='x', rotation=15)
 
fig.suptitle('Imbalance Technique Comparison — Balanced Accuracy & F1-Score',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()
 
# ROC overlay — all four
plt.figure(figsize=(9, 6))
line_colors = ['#AAAAAA', '#FF6B6B', '#4ECDC4', '#FFD93D']
for (name, model), col in zip(
    [('Baseline (no resample)', model_nores), ('TomekLinks', model_tl),
     ('SMOTE', model_smt),                    ('SMOTETomek', model_st)],
    line_colors
):
    proba = model.predict_proba(X_te_eng)[:, 1]
    fpr, tpr, _ = roc_curve(y_test_rf, proba)
    auc = roc_auc_score(y_test_rf, proba)
    plt.plot(fpr, tpr, lw=2, color=col, label=f'{name} (AUC={auc:.3f})')
plt.plot([0,1],[0,1],'--', color='grey', label='Random')
plt.title('ROC Overlay — Baseline vs Imbalance Techniques', fontweight='bold')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
 
# ── Automatically select the best technique ───────────────────
resample_only = {k: v for k, v in technique_results.items()
                 if k != 'Baseline (no resample)'}
 
best_technique = max(
    resample_only,
    key=lambda k: (resample_only[k]['Balanced Accuracy'],
                   resample_only[k]['F1-Score'])
)
 
X_best, y_best = resample_data[best_technique]
 
print(f'\n{"="*60}')
print(f'  ✅ BEST IMBALANCE TECHNIQUE: {best_technique}')
print(f'     Balanced Accuracy : {technique_results[best_technique]["Balanced Accuracy"]:.4f}')
print(f'     F1-Score          : {technique_results[best_technique]["F1-Score"]:.4f}')
print(f'     ROC-AUC           : {technique_results[best_technique]["ROC-AUC"]:.4f}')
print(f'\n  → Used for Tuned RF in Step 7.')
print(f'{"="*60}')

In [ ]:
# ─────────────────────────────────────────────────────────────
# STEP 7 — HYPERPARAMETER-TUNED RANDOM FOREST
#           (same best resampling technique)
# ─────────────────────────────────────────────────────────────

# Number of random combinations to try
n_iter   = 30

# Number of cross-validation folds
cv_folds = 3

# ══════════════════════════════════════════════════════════════
# ▼▼▼  USER-CONFIGURABLE HYPERPARAMETER SEARCH SETTINGS  ▼▼▼
# ══════════════════════════════════════════════════════════════
param_dist = {
    'n_estimators'     : [300],          # fixed: 300
    'max_depth'        : [10],           # fixed: 10
    'min_samples_split': [2],            # fixed: 2
    'min_samples_leaf' : [2],            # fixed: 2
    'max_features'     : ['log2'],       # fixed: log2
    'class_weight'     : ['balanced'],   # fixed: balanced
}
# ▲▲▲  END OF USER-CONFIGURABLE SECTION  ▲▲▲
# ══════════════════════════════════════════════════════════════

print('\n' + '='*60)
print(f'  STEP 7 — HYPERPARAMETER TUNING ({best_technique})')
print('='*60)
print(f'  Search space summary:')
for param, values in param_dist.items():
    print(f'   {param:<22}: {values}')
print(f'\n  n_iter   = {n_iter}  (random combinations to try)')
print(f'  cv_folds = {cv_folds}  (stratified cross-validation folds)')
print(f'  scoring  = roc_auc')

cv_strategy = StratifiedKFold(n_splits=cv_folds, shuffle=True,
                              random_state=RANDOM_STATE)

rscv = RandomizedSearchCV(
    estimator           = RandomForestClassifier(random_state=RANDOM_STATE,
                                                 n_jobs=-1),
    param_distributions = param_dist,
    n_iter              = n_iter,
    scoring             = 'roc_auc',
    cv                  = cv_strategy,
    verbose             = 1,
    random_state        = RANDOM_STATE,
    n_jobs              = -1,
    refit               = True
)

rscv.fit(X_best, y_best)

print(f'\n✅ Best parameters found:')
for k, v in rscv.best_params_.items():
    print(f'   {k:<22}: {v}')
print(f'\n   Best CV ROC-AUC: {rscv.best_score_:.4f}')

rf_tuned = rscv.best_estimator_

y_tu_val_pred   = rf_tuned.predict(X_vl_eng)
y_tu_val_proba  = rf_tuned.predict_proba(X_vl_eng)[:, 1]
y_tu_test_pred  = rf_tuned.predict(X_te_eng)
y_tu_test_proba = rf_tuned.predict_proba(X_te_eng)[:, 1]

tuned_metrics = {
    'Accuracy'         : accuracy_score(y_test,  y_tu_test_pred),
    'Balanced Accuracy': balanced_accuracy_score(y_test, y_tu_test_pred),
    'F1-Score'         : f1_score(y_test,  y_tu_test_pred),
    'ROC-AUC'          : roc_auc_score(y_test, y_tu_test_proba),
}
tuned_val_metrics = {
    'Accuracy'         : accuracy_score(y_val,  y_tu_val_pred),
    'Balanced Accuracy': balanced_accuracy_score(y_val, y_tu_val_pred),
    'F1-Score'         : f1_score(y_val,  y_tu_val_pred),
    'ROC-AUC'          : roc_auc_score(y_val, y_tu_val_proba),
}

print(f'\nTUNED RF — {best_technique}')
print('=' * 60)
print(f'{"Metric":<22} {"Validation":>12} {"Test":>12}')
print('-'*48)
for k in tuned_metrics:
    print(f'{k:<22} {tuned_val_metrics[k]:>12.4f} {tuned_metrics[k]:>12.4f}')

print(f'\nTest Classification Report:')
print(classification_report(y_test, y_tu_test_pred,
                            target_names=['No (0)', 'Yes (1)']))

cm_tu        = confusion_matrix(y_test, y_tu_test_pred)
fpr_tu, tpr_tu, _ = roc_curve(y_test, y_tu_test_proba)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cm_tu, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Pred No', 'Pred Yes'],
            yticklabels=['Act No',  'Act Yes'])
axes[0].set_title(f'Confusion Matrix — Tuned RF ({best_technique})', fontweight='bold')
axes[1].plot(fpr_tu, tpr_tu, color='#4ECDC4', lw=2.5,
             label=f'Tuned RF (AUC={tuned_metrics["ROC-AUC"]:.3f})')
axes[1].plot([0,1],[0,1],'--', color='grey', label='Random')
axes[1].set_title(f'ROC Curve — Tuned RF ({best_technique})', fontweight='bold')
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

feat_imp = pd.Series(rf_tuned.feature_importances_,
                     index=X_tr_eng.columns).sort_values(ascending=False)
plt.figure(figsize=(10, 6))
feat_imp.head(20).plot(kind='bar', color='#4ECDC4', edgecolor='white')
plt.title('Top 20 Feature Importances — Tuned RF', fontweight='bold')
plt.ylabel('Mean Decrease in Impurity')
plt.xticks(rotation=45, ha='right')
plt.tight_layout(); plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# STEP 8 — BASELINE vs TUNED RF COMPARISON
# ─────────────────────────────────────────────────────────────
# Both models use the same best resampling technique.
# This comparison isolates the effect of hyperparameter tuning.
 
print('\n' + '='*60)
print(f'  STEP 8 — BASELINE vs TUNED RF ({best_technique})')
print('='*60)
print('  Baseline RF : Step 4 result (no resampling, no tuning)')
print(f'  Tuned RF    : Step 7 result ({best_technique} + hyperparameter tuning)')
print(f'{"Metric":<22} {"Baseline RF":>13} {"Tuned RF":>12} {"Delta":>10}')
print('-'*60)
for k in m_no_resample:
    delta = tuned_metrics[k] - m_no_resample[k]
    arrow = '▲' if delta > 0 else ('▼' if delta < 0 else '–')
    print(f'{k:<22} {m_no_resample[k]:>13.4f} {tuned_metrics[k]:>12.4f} '
          f'{arrow}{abs(delta):>8.4f}')
 
# ROC overlay
fpr_bl, tpr_bl, _ = roc_curve(y_test_rf, model_nores.predict_proba(X_te_eng)[:, 1])
plt.figure(figsize=(8, 6))
plt.plot(fpr_bl, tpr_bl, color='#FF6B6B', lw=2,
         label=f'Baseline RF (AUC={m_no_resample["ROC-AUC"]:.3f})')
plt.plot(fpr_tu, tpr_tu, color='#4ECDC4', lw=2,
         label=f'Tuned RF    (AUC={tuned_metrics["ROC-AUC"]:.3f})')
plt.plot([0,1],[0,1],'--', color='grey', label='Random')
plt.title(f'ROC — Baseline vs Tuned RF ({best_technique})', fontweight='bold')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
 
# Confusion matrix side by side
cm_bl_step4 = confusion_matrix(y_test_rf, model_nores.predict(X_te_eng))
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, cm, label in [(axes[0], cm_bl_step4, 'Baseline RF (Step 4)'),
                       (axes[1], cm_tu,        'Tuned RF (Step 7)')]:
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Pred No', 'Pred Yes'],
                yticklabels=['Act No',  'Act Yes'])
    ax.set_title(f'{label}', fontweight='bold')
fig.suptitle('Confusion Matrix Comparison — Baseline vs Tuned RF',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()
 
print('\n' + '='*60)
print('  FINAL SUMMARY')
print('='*60)
print(f'  Best imbalance technique : {best_technique}')
print(f'  Baseline RF — Bal.Acc    : {m_no_resample["Balanced Accuracy"]:.4f} | '
      f'F1: {m_no_resample["F1-Score"]:.4f} | AUC: {m_no_resample["ROC-AUC"]:.4f}')
print(f'  Tuned RF    — Bal.Acc    : {tuned_metrics["Balanced Accuracy"]:.4f} | '
      f'F1: {tuned_metrics["F1-Score"]:.4f} | AUC: {tuned_metrics["ROC-AUC"]:.4f}')
improvement = tuned_metrics["ROC-AUC"] - m_no_resample["ROC-AUC"]
direction   = "improvement" if improvement >= 0 else "degradation"
print(f'  Tuning {direction}         : {improvement:+.4f} AUC')
print('='*60)

In [ ]:
# =============================================================
# MILESTONE 3 — XGBOOST PIPELINE
# ============================================================

# ─────────────────────────────────────────────────────────────
# STEP 0 — IMPORT NEW LIBRARIES (for Milestone 3)
# ─────────────────────────────────────────────────────────────
# All standard libraries (pandas, numpy, sklearn, matplotlib, seaborn)
# are already imported in Cell 03. Only new libraries are imported here.

from xgboost import XGBClassifier

from imblearn.under_sampling import TomekLinks
from imblearn.over_sampling  import SMOTE
from imblearn.combine        import SMOTETomek

print('✅ XGBoost + imbalanced-learn libraries imported')

In [ ]:
# ─────────────────────────────────────────────────────────────
# STEP 1 — BASELINE XGBOOST
#           (no resampling, no hyperparameter tuning)
# ─────────────────────────────────────────────────────────────
# We first establish a baseline using the raw (unbalanced)
# training set with default hyperparameters.
# scale_pos_weight is set to the majority/minority class ratio
# to give XGBoost a soft awareness of imbalance without any
# resampling — equivalent to class_weight='balanced' in RF.
# This result becomes the reference point for comparing the
# three imbalance techniques in Step 5.

# ── Compute scale_pos_weight for the imbalanced training set ──
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
SPW       = neg_count / pos_count   # ~9 for an 11% positive rate

# ── Helper: train one XGBoost variant and return metrics + model ──
def evaluate_xgb(X_res, y_res, params, label, show_plots=True):
    """
    Trains an XGBClassifier on (X_res, y_res).
    Evaluates on the fixed val and test sets.
    Returns (test_metrics_dict, fitted_model).
    """
    model = XGBClassifier(**params)
    model.fit(X_res, y_res, eval_set=[(X_vl_eng, y_val)], verbose=False)

    y_val_pred   = model.predict(X_vl_eng)
    y_val_proba  = model.predict_proba(X_vl_eng)[:, 1]
    y_test_pred  = model.predict(X_te_eng)
    y_test_proba = model.predict_proba(X_te_eng)[:, 1]

    def get_metrics(y_true, y_pred, y_proba):
        return {
            'Accuracy'         : accuracy_score(y_true, y_pred),
            'Balanced Accuracy': balanced_accuracy_score(y_true, y_pred),
            'F1-Score'         : f1_score(y_true, y_pred),
            'ROC-AUC'          : roc_auc_score(y_true, y_proba),
        }

    val_m  = get_metrics(y_val,  y_val_pred,  y_val_proba)
    test_m = get_metrics(y_test, y_test_pred, y_test_proba)

    print(f'\n{"="*60}')
    print(f'  {label}')
    print(f'{"="*60}')
    print(f'{"Metric":<22} {"Validation":>12} {"Test":>12}')
    print(f'{"-"*48}')
    for k in val_m:
        print(f'{k:<22} {val_m[k]:>12.4f} {test_m[k]:>12.4f}')
    print(f'\nTest Classification Report:')
    print(classification_report(y_test, y_test_pred,
                                target_names=['No (0)', 'Yes (1)']))

    if show_plots:
        cm = confusion_matrix(y_test, y_test_pred)
        fpr, tpr, _ = roc_curve(y_test, y_test_proba)
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
                    xticklabels=['Pred No', 'Pred Yes'],
                    yticklabels=['Act No',  'Act Yes'])
        axes[0].set_title(f'Confusion Matrix — {label}', fontweight='bold')
        axes[1].plot(fpr, tpr, color='#4ECDC4', lw=2.5,
                     label=f'{label} (AUC={test_m["ROC-AUC"]:.3f})')
        axes[1].plot([0,1],[0,1],'--', color='#FF6B6B', label='Random')
        axes[1].set_title(f'ROC Curve — {label}', fontweight='bold')
        axes[1].set_xlabel('False Positive Rate')
        axes[1].set_ylabel('True Positive Rate')
        axes[1].legend(); axes[1].grid(True, alpha=0.3)
        plt.tight_layout(); plt.show()

    return test_m, model

# ── Default XGBoost params (reused across Steps 4 and 5) ─────
DEFAULT_PARAMS = dict(
    n_estimators      = 100,
    max_depth         = 6,           # XGBoost default 
    learning_rate     = 0.1,         # XGBoost-specific: shrinks each tree's contribution
    scale_pos_weight  = SPW,        
    use_label_encoder = False,
    eval_metric       = 'logloss',
    random_state      = RANDOM_STATE,
    n_jobs            = -1
)

# When resampling is applied, classes are already balanced → reset to 1
RESAMPLE_PARAMS = {**DEFAULT_PARAMS, 'scale_pos_weight': 1}

print('\n' + '='*60)
print('  STEP 1 — BASELINE XGBoost (no resampling, no tuning)')
print(f'           scale_pos_weight = {SPW:.2f}  (neg/pos ratio)')
print('='*60)

m_no_resample, model_nores = evaluate_xgb(
    X_tr_eng, y_train, DEFAULT_PARAMS,
    label='Baseline XGBoost (no resampling)'
)


In [ ]:
# ─────────────────────────────────────────────────────────────
# STEP 2A — IMBALANCE HANDLING (compare 3 techniques)
# ─────────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════
# TECHNIQUE A — TOMEK LINKS (UNDERSAMPLING)
# ══════════════════════════════════════════════════════════════
print('\n' + '='*60)
print('  STEP 2: TECHNIQUE A — TOMEK LINKS (UNDERSAMPLING)')
print('='*60)

tl          = TomekLinks()
X_tl, y_tl = tl.fit_resample(X_tr_eng, y_train)
removed     = len(y_train) - len(y_tl)
print(f'Tomek Links removed {removed:,} borderline majority samples')
print(f'After resampling → {X_tl.shape[0]:,} samples | positive rate: {y_tl.mean():.3f}')

m_tl, model_tl = evaluate_xgb(
    X_tl, y_tl, RESAMPLE_PARAMS,
    label='Technique A — TomekLinks'
)

# Before vs After — Technique A
print('\nTECHNIQUE A — Baseline vs TomekLinks')
print(f'{"Metric":<22} {"Baseline":>13} {"TomekLinks":>12} {"Delta":>10}')
print('-'*60)
for k in m_no_resample:
    delta = m_tl[k] - m_no_resample[k]
    arrow = '▲' if delta > 0 else ('▼' if delta < 0 else '–')
    print(f'{k:<22} {m_no_resample[k]:>13.4f} {m_tl[k]:>12.4f} '
          f'{arrow}{abs(delta):>8.4f}')



In [ ]:
# ─────────────────────────────────────────────────────────────
# STEP 2B — IMBALANCE HANDLING (compare 3 techniques)
# ─────────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════
# TECHNIQUE B — SMOTE (OVERSAMPLING)
# ══════════════════════════════════════════════════════════════
print('\n' + '='*60)
print('  STEP 2: TECHNIQUE B — SMOTE (OVERSAMPLING)')
print('='*60)

smote         = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
X_smt, y_smt = smote.fit_resample(X_tr_eng, y_train)
added         = len(y_smt) - len(y_train)
print(f'SMOTE synthesised {added:,} new minority samples')
print(f'After resampling → {X_smt.shape[0]:,} samples | positive rate: {y_smt.mean():.3f}')

m_smt, model_smt = evaluate_xgb(
    X_smt, y_smt, RESAMPLE_PARAMS,
    label='Technique B — SMOTE'
)

# Before vs After — Technique B
print('\nTECHNIQUE B — Baseline vs SMOTE')
print(f'{"Metric":<22} {"Baseline":>13} {"SMOTE":>12} {"Delta":>10}')
print('-'*60)
for k in m_no_resample:
    delta = m_smt[k] - m_no_resample[k]
    arrow = '▲' if delta > 0 else ('▼' if delta < 0 else '–')
    print(f'{k:<22} {m_no_resample[k]:>13.4f} {m_smt[k]:>12.4f} '
          f'{arrow}{abs(delta):>8.4f}')


In [ ]:
# ─────────────────────────────────────────────────────────────
# STEP 2C — IMBALANCE HANDLING (compare 3 techniques)
# ─────────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════
# TECHNIQUE C — SMOTETomek (HYBRID)
# ══════════════════════════════════════════════════════════════
print('\n' + '='*60)
print('  STEP 2: TECHNIQUE C — SMOTETomek (HYBRID)')
print('='*60)

st_hybrid    = SMOTETomek(random_state=RANDOM_STATE)
X_st, y_st   = st_hybrid.fit_resample(X_tr_eng, y_train)
print(f'SMOTETomek → {X_st.shape[0]:,} samples | positive rate: {y_st.mean():.3f}')

m_st, model_st = evaluate_xgb(
    X_st, y_st, RESAMPLE_PARAMS,
    label='Technique C — SMOTETomek'
)

# Before vs After — Technique C
print('\nTECHNIQUE C — Baseline vs SMOTETomek')
print(f'{"Metric":<22} {"Baseline":>13} {"SMOTETomek":>12} {"Delta":>10}')
print('-'*60)
for k in m_no_resample:
    delta = m_st[k] - m_no_resample[k]
    arrow = '▲' if delta > 0 else ('▼' if delta < 0 else '–')
    print(f'{k:<22} {m_no_resample[k]:>13.4f} {m_st[k]:>12.4f} '
          f'{arrow}{abs(delta):>8.4f}')

In [ ]:
# ─────────────────────────────────────────────────────────────
# STEP 3 — CHOOSE BEST IMBALANCE TECHNIQUE
# ─────────────────────────────────────────────────────────────
# We rank the three techniques by Balanced Accuracy (primary)
# and F1-Score (tiebreaker).
# These two metrics are prioritised over raw Accuracy and
# ROC-AUC because the dataset is heavily imbalanced (~9:1).
# Raw Accuracy is dominated by the majority class and can be
# misleading. Balanced Accuracy gives equal weight to both
# classes. F1-Score captures the precision-recall trade-off
# which is critical for correctly identifying subscribers.

technique_results = {
    'Baseline (no resample)': m_no_resample,
    'TomekLinks'            : m_tl,
    'SMOTE'                 : m_smt,
    'SMOTETomek'            : m_st,
}

resample_data = {
    'Baseline (no resample)': (X_tr_eng, y_train),   # ← added
    'TomekLinks'            : (X_tl,     y_tl),
    'SMOTE'                 : (X_smt,    y_smt),
    'SMOTETomek'            : (X_st,     y_st),
}

print('\n' + '='*70)
print('  STEP 3 — IMBALANCE TECHNIQUE COMPARISON (same Default XGBoost)')
print('='*70)
print(f'{"Technique":<24} {"Accuracy":>10} {"Bal.Acc":>10} '
      f'{"F1-Score":>10} {"ROC-AUC":>10}')
print('-'*66)
for name, m in technique_results.items():
    marker = ' ← baseline' if name == 'Baseline (no resample)' else ''
    print(f'{name:<24} {m["Accuracy"]:>10.4f} {m["Balanced Accuracy"]:>10.4f} '
          f'{m["F1-Score"]:>10.4f} {m["ROC-AUC"]:>10.4f}{marker}')

# Visual comparison — bar chart (Balanced Accuracy & F1-Score)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
names   = list(technique_results.keys())
colors  = ['#AAAAAA', '#FF6B6B', '#4ECDC4', '#FFD93D']

for ax, metric in zip(axes, ['Balanced Accuracy', 'F1-Score']):
    vals = [technique_results[n][metric] for n in names]
    bars = ax.bar(names, vals, color=colors, edgecolor='white', width=0.5)
    ax.set_title(metric, fontweight='bold')
    ax.set_ylabel(metric)
    ax.set_ylim(min(vals) - 0.05, max(vals) + 0.05)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.002,
                f'{val:.4f}', ha='center', fontsize=9, fontweight='bold')
    ax.grid(True, alpha=0.2)
    ax.tick_params(axis='x', rotation=15)

fig.suptitle('Imbalance Technique Comparison — Balanced Accuracy & F1-Score',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

# ROC overlay — all four
plt.figure(figsize=(9, 6))
line_colors = ['#AAAAAA', '#FF6B6B', '#4ECDC4', '#FFD93D']
for (name, model), col in zip(
    [('Baseline (no resample)', model_nores), ('TomekLinks', model_tl),
     ('SMOTE', model_smt),                    ('SMOTETomek', model_st)],
    line_colors
):
    proba = model.predict_proba(X_te_eng)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    plt.plot(fpr, tpr, lw=2, color=col, label=f'{name} (AUC={auc:.3f})')
plt.plot([0,1],[0,1],'--', color='grey', label='Random')
plt.title('ROC Overlay — Baseline vs Imbalance Techniques', fontweight='bold')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# ── Automatically select the best technique ───────────────────
# ⚠️  CHANGED: baseline is now included in selection pool
# Previously excluded — which forced a resampling method to always win
# even when baseline outperformed all three techniques.
best_technique = max(
    technique_results,                          # ← was resample_only, now ALL four
    key=lambda k: (technique_results[k]['Balanced Accuracy'],
                   technique_results[k]['F1-Score'])
)

X_best, y_best = resample_data[best_technique]

# ── Set scale_pos_weight correctly based on winner ───────────
# If baseline wins → keep SPW high (~9) to correct real imbalance
# If resampling wins → SPW=1 since data is already balanced
if best_technique == 'Baseline (no resample)':
    spw_best = SPW
else:
    spw_best = (y_best == 0).sum() / (y_best == 1).sum()

print(f'\n{"="*60}')
print(f'  ✅ BEST TECHNIQUE (baseline eligible): {best_technique}')
print(f'     Balanced Accuracy : {technique_results[best_technique]["Balanced Accuracy"]:.4f}')
print(f'     F1-Score          : {technique_results[best_technique]["F1-Score"]:.4f}')
print(f'     ROC-AUC           : {technique_results[best_technique]["ROC-AUC"]:.4f}')
print(f'     scale_pos_weight  : {spw_best:.2f}')
print(f'\n  → Used for Tuned XGBoost in Step 7.')
print(f'{"="*60}')

In [ ]:
# ─────────────────────────────────────────────────────────────
# STEP 4 — TUNE scale_pos_weight (since Baseline won)
# ─────────────────────────────────────────────────────────────
print('\n' + '='*60)
print('  STEP 4 — scale_pos_weight SWEEP')
print('='*60)

spw_candidates = [SPW * 0.5, SPW * 0.75, SPW, SPW * 1.25, SPW * 1.5, SPW * 2.0]
spw_results = []

for spw_val in spw_candidates:
    model_spw = XGBClassifier(
        n_estimators          = 200,
        max_depth             = 7,
        learning_rate         = 0.01,
        subsample             = 0.9,
        colsample_bytree      = 0.9,
        min_child_weight      = 3,
        gamma                 = 0.1,
        reg_alpha             = 0.1,
        reg_lambda            = 1.5,
        scale_pos_weight      = spw_val,       # ← was wrongly spw_best, now spw_val
        early_stopping_rounds = 30,
        use_label_encoder     = False,
        eval_metric           = 'logloss',     # ← changed from 'auc' (logloss works better with early stopping + bal.acc goal)
        random_state          = RANDOM_STATE,
        n_jobs                = -1
    )
    model_spw.fit(
        X_tr_eng, y_train,
        eval_set  = [(X_vl_eng, y_val)],       # ← this was missing, caused the error
        verbose   = False
    )
    y_val_pred = model_spw.predict(X_vl_eng)
    bal_acc    = balanced_accuracy_score(y_val, y_val_pred)
    f1         = f1_score(y_val, y_val_pred)
    spw_results.append({'spw': spw_val, 'Bal.Acc': bal_acc, 'F1': f1, 'model': model_spw})
    print(f'  SPW={spw_val:6.2f} → Val Bal.Acc: {bal_acc:.4f} | F1: {f1:.4f}')

# Pick the best SPW based on validation Balanced Accuracy
best_spw_row = max(spw_results, key=lambda x: (x['Bal.Acc'], x['F1']))
spw_best     = best_spw_row['spw']

print(f'\n  ✅ Best scale_pos_weight: {spw_best:.2f}')
print(f'     Val Bal.Acc: {best_spw_row["Bal.Acc"]:.4f} | F1: {best_spw_row["F1"]:.4f}')
print(f'  → spw_best updated. Step 7 will use this value.')

In [ ]:
# ─────────────────────────────────────────────────────────────
# STEP 5 — TUNED XGBOOST
#           (lecturer-style: manually chosen robust parameters
#            + early stopping instead of RandomizedSearchCV)
# ─────────────────────────────────────────────────────────────

print('\n' + '='*60)
print(f'  STEP 5 — TUNED XGBoost ({best_technique})')
print('='*60)

# spw_best was determined by the Step 4 sweep — used directly here
print(f'  scale_pos_weight (from Step 4 sweep): {spw_best:.2f}')
print(f'  Resampled set: {X_best.shape[0]:,} samples | positive rate: {y_best.mean():.3f}')

# ── Tuned parameters (robust, regularized, early-stopping) ───
xgb_tuned = XGBClassifier(
    n_estimators      = 200,       # high value — early stopping will find the best
    max_depth         = 7,         # slightly shallower than lecturer (bank data is different)
    learning_rate     = 0.01,      # slow learning = better generalisation
    subsample         = 0.9,       # row subsampling per tree
    colsample_bytree  = 0.9,       # feature subsampling per tree
    min_child_weight  = 3,         # prevents splits on very few samples
    gamma             = 0.1,       # minimum loss reduction to split
    reg_alpha         = 0.1,       # L1 regularization — prevents overfitting
    reg_lambda        = 1.5,       # L2 regularization — prevents overfitting
    scale_pos_weight  = spw_best,  # handles remaining imbalance
    early_stopping_rounds = 30,    # stop if val logloss doesn't improve for 30 rounds
    use_label_encoder = False,
    eval_metric       = 'auc',
    random_state      = RANDOM_STATE,
    n_jobs            = -1
)

# Early stopping on validation set — automatically finds best n_estimators
xgb_tuned.fit(
    X_best, y_best,
    eval_set        = [(X_vl_eng, y_val)],
    verbose         = 50,   # print every 50 rounds
)

best_iteration = xgb_tuned.best_iteration
print(f'\n✅ Best iteration (early stopping): {best_iteration}')

# ── Evaluate ──────────────────────────────────────────────────
y_tu_val_pred   = xgb_tuned.predict(X_vl_eng)
y_tu_val_proba  = xgb_tuned.predict_proba(X_vl_eng)[:, 1]
y_tu_test_pred  = xgb_tuned.predict(X_te_eng)
y_tu_test_proba = xgb_tuned.predict_proba(X_te_eng)[:, 1]

tuned_metrics = {
    'Accuracy'         : accuracy_score(y_test,  y_tu_test_pred),
    'Balanced Accuracy': balanced_accuracy_score(y_test, y_tu_test_pred),
    'F1-Score'         : f1_score(y_test,  y_tu_test_pred),
    'ROC-AUC'          : roc_auc_score(y_test, y_tu_test_proba),
}
tuned_val_metrics = {
    'Accuracy'         : accuracy_score(y_val,  y_tu_val_pred),
    'Balanced Accuracy': balanced_accuracy_score(y_val, y_tu_val_pred),
    'F1-Score'         : f1_score(y_val,  y_tu_val_pred),
    'ROC-AUC'          : roc_auc_score(y_val, y_tu_val_proba),
}

print(f'\nTUNED XGBoost — {best_technique}')
print('=' * 60)
print(f'{"Metric":<22} {"Validation":>12} {"Test":>12}')
print('-'*48)
for k in tuned_metrics:
    print(f'{k:<22} {tuned_val_metrics[k]:>12.4f} {tuned_metrics[k]:>12.4f}')

print(f'\nTest Classification Report:')
print(classification_report(y_test, y_tu_test_pred,
                            target_names=['No (0)', 'Yes (1)']))

# ── Plots ─────────────────────────────────────────────────────
cm_tu             = confusion_matrix(y_test, y_tu_test_pred)
fpr_tu, tpr_tu, _ = roc_curve(y_test, y_tu_test_proba)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cm_tu, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Pred No', 'Pred Yes'],
            yticklabels=['Act No',  'Act Yes'])
axes[0].set_title(f'Confusion Matrix — Tuned XGBoost ({best_technique})', fontweight='bold')
axes[1].plot(fpr_tu, tpr_tu, color='#4ECDC4', lw=2.5,
             label=f'Tuned XGBoost (AUC={tuned_metrics["ROC-AUC"]:.3f})')
axes[1].plot([0,1],[0,1],'--', color='grey', label='Random')
axes[1].set_title(f'ROC Curve — Tuned XGBoost ({best_technique})', fontweight='bold')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# ── Feature importance ────────────────────────────────────────
feat_imp = pd.Series(xgb_tuned.feature_importances_,
                     index=X_tr_eng.columns).sort_values(ascending=False)
plt.figure(figsize=(10, 6))
feat_imp.head(20).plot(kind='bar', color='#4ECDC4', edgecolor='white')
plt.title('Top 20 Feature Importances — Tuned XGBoost (gain)', fontweight='bold')
plt.ylabel('Feature Importance Score')
plt.xticks(rotation=45, ha='right')
plt.tight_layout(); plt.show()


In [ ]:
# ─────────────────────────────────────────────────────────────
# STEP 6A — GRIDSEARCHCV (exhaustive search, smaller grid)
# ─────────────────────────────────────────────────────────────
# GridSearch tries every combination — so keep the grid small
# to avoid running for hours. We search around the best manual
# params found in Step 5 as a starting point.

from sklearn.model_selection import GridSearchCV, StratifiedKFold

print('\n' + '='*60)
print('  STEP 6A — GridSearchCV')
print('='*60)

grid_param = {
    'max_depth'        : [5, 6, 7],
    'learning_rate'    : [0.01, 0.05, 0.1],
    'subsample'        : [0.8, 0.9],
    'colsample_bytree' : [0.8, 0.9],
    'min_child_weight' : [1, 3],
}
# Total combinations: 3×3×2×2×2 = 72 fits × 5 folds = 360 fits (manageable)

cv_gs = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

gs = GridSearchCV(
    XGBClassifier(
        n_estimators      = 200,
        gamma             = 0.1,
        reg_alpha         = 0.1,
        reg_lambda        = 1.5,
        scale_pos_weight  = spw_best,
        use_label_encoder = False,
        eval_metric       = 'logloss',
        random_state      = RANDOM_STATE,
        n_jobs            = -1
    ),
    param_grid  = grid_param,
    scoring     = 'balanced_accuracy',
    cv          = cv_gs,
    n_jobs      = -1,
    verbose     = 1
)

gs.fit(X_tr_eng, y_train)

print(f'\n✅ GridSearch best params: {gs.best_params_}')
print(f'   CV Balanced Accuracy  : {gs.best_score_:.4f}')

xgb_gs = gs.best_estimator_

# Evaluate on val and test
y_gs_val_pred   = xgb_gs.predict(X_vl_eng)
y_gs_val_proba  = xgb_gs.predict_proba(X_vl_eng)[:, 1]
y_gs_test_pred  = xgb_gs.predict(X_te_eng)
y_gs_test_proba = xgb_gs.predict_proba(X_te_eng)[:, 1]

gs_metrics = {
    'Accuracy'         : accuracy_score(y_test, y_gs_test_pred),
    'Balanced Accuracy': balanced_accuracy_score(y_test, y_gs_test_pred),
    'F1-Score'         : f1_score(y_test, y_gs_test_pred),
    'ROC-AUC'          : roc_auc_score(y_test, y_gs_test_proba),
}
gs_val_metrics = {
    'Accuracy'         : accuracy_score(y_val, y_gs_val_pred),
    'Balanced Accuracy': balanced_accuracy_score(y_val, y_gs_val_pred),
    'F1-Score'         : f1_score(y_val, y_gs_val_pred),
    'ROC-AUC'          : roc_auc_score(y_val, y_gs_val_proba),
}

print(f'\n{"Metric":<22} {"Validation":>12} {"Test":>12}')
print('-'*48)
for k in gs_metrics:
    print(f'{k:<22} {gs_val_metrics[k]:>12.4f} {gs_metrics[k]:>12.4f}')

print(f'\nTest Classification Report (GridSearch):')
print(classification_report(y_test, y_gs_test_pred, target_names=['No (0)', 'Yes (1)']))

In [ ]:
# ─────────────────────────────────────────────────────────────
# STEP 6B — RANDOMIZEDSEARCHCV (wide search, more params)
# ─────────────────────────────────────────────────────────────
# Randomly samples n_iter combinations from a wider grid.
# More efficient than GridSearch when the grid is large.
# We include scale_pos_weight in the search as well.

from sklearn.model_selection import RandomizedSearchCV

print('\n' + '='*60)
print('  STEP 6B — RandomizedSearchCV')
print('='*60)

rand_param = {
    'max_depth'        : [3, 4, 5, 6, 7, 8],
    'learning_rate'    : [0.005, 0.01, 0.02, 0.05, 0.1, 0.2],
    'n_estimators'     : [100, 200, 300, 500],
    'subsample'        : [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree' : [0.6, 0.7, 0.8, 0.9, 1.0],
    'min_child_weight' : [1, 2, 3, 5, 10],
    'gamma'            : [0, 0.01, 0.05, 0.1, 0.2, 0.5],
    'reg_alpha'        : [0, 0.01, 0.1, 0.5, 1.0],
    'reg_lambda'       : [0.5, 1.0, 1.5, 2.0, 5.0],
    'scale_pos_weight' : [SPW * 0.5, SPW * 0.75, SPW, SPW * 1.25, SPW * 1.5],
}

cv_rs = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rs = RandomizedSearchCV(
    XGBClassifier(
        use_label_encoder = False,
        eval_metric       = 'logloss',
        random_state      = RANDOM_STATE,
        n_jobs            = -1
    ),
    param_distributions = rand_param,
    n_iter              = 100,               # 100 random combos × 5 folds = 500 fits
    scoring             = 'balanced_accuracy',
    cv                  = cv_rs,
    random_state        = RANDOM_STATE,
    n_jobs              = -1,
    verbose             = 1
)

rs.fit(X_tr_eng, y_train)

print(f'\n✅ RandomizedSearch best params: {rs.best_params_}')
print(f'   CV Balanced Accuracy        : {rs.best_score_:.4f}')

xgb_rs = rs.best_estimator_

# Evaluate
y_rs_val_pred   = xgb_rs.predict(X_vl_eng)
y_rs_val_proba  = xgb_rs.predict_proba(X_vl_eng)[:, 1]
y_rs_test_pred  = xgb_rs.predict(X_te_eng)
y_rs_test_proba = xgb_rs.predict_proba(X_te_eng)[:, 1]

rs_metrics = {
    'Accuracy'         : accuracy_score(y_test, y_rs_test_pred),
    'Balanced Accuracy': balanced_accuracy_score(y_test, y_rs_test_pred),
    'F1-Score'         : f1_score(y_test, y_rs_test_pred),
    'ROC-AUC'          : roc_auc_score(y_test, y_rs_test_proba),
}
rs_val_metrics = {
    'Accuracy'         : accuracy_score(y_val, y_rs_val_pred),
    'Balanced Accuracy': balanced_accuracy_score(y_val, y_rs_val_pred),
    'F1-Score'         : f1_score(y_val, y_rs_val_pred),
    'ROC-AUC'          : roc_auc_score(y_val, y_rs_val_proba),
}

print(f'\n{"Metric":<22} {"Validation":>12} {"Test":>12}')
print('-'*48)
for k in rs_metrics:
    print(f'{k:<22} {rs_val_metrics[k]:>12.4f} {rs_metrics[k]:>12.4f}')

print(f'\nTest Classification Report (RandomizedSearch):')
print(classification_report(y_test, y_rs_test_pred, target_names=['No (0)', 'Yes (1)']))

In [ ]:
# ─────────────────────────────────────────────────────────────
# STEP 6C — BAYESIAN OPTIMIZATION (most efficient search)
# ─────────────────────────────────────────────────────────────
# Uses past trial results to intelligently choose next params.
# Finds good solutions faster than Grid or Random search.
# Requires: pip install scikit-optimize

import subprocess
subprocess.run(['pip', 'install', 'scikit-optimize'], check=True)

from skopt import BayesSearchCV
from skopt.space import Real, Integer

print('\n' + '='*60)
print('  STEP 6C — Bayesian Optimization (BayesSearchCV)')
print('='*60)

bayes_param = {
    'max_depth'        : Integer(3, 8),
    'learning_rate'    : Real(0.005, 0.2, prior='log-uniform'),
    'n_estimators'     : Integer(100, 500),
    'subsample'        : Real(0.6, 1.0),
    'colsample_bytree' : Real(0.6, 1.0),
    'min_child_weight' : Integer(1, 10),
    'gamma'            : Real(0, 0.5),
    'reg_alpha'        : Real(0, 1.0),
    'reg_lambda'       : Real(0.5, 5.0),
    'scale_pos_weight' : Real(SPW * 0.5, SPW * 2.0),
}

cv_bayes = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

bayes = BayesSearchCV(
    XGBClassifier(
        use_label_encoder = False,
        eval_metric       = 'logloss',
        random_state      = RANDOM_STATE,
        n_jobs            = -1
    ),
    search_spaces = bayes_param,
    n_iter        = 50,                  # 50 smart trials × 5 folds = 250 fits
    scoring       = 'balanced_accuracy',
    cv            = cv_bayes,
    random_state  = RANDOM_STATE,
    n_jobs        = -1,
    verbose       = 1
)

bayes.fit(X_tr_eng, y_train)

print(f'\n✅ Bayesian best params : {dict(bayes.best_params_)}')
print(f'   CV Balanced Accuracy : {bayes.best_score_:.4f}')

xgb_bayes = bayes.best_estimator_

# Evaluate
y_by_val_pred   = xgb_bayes.predict(X_vl_eng)
y_by_val_proba  = xgb_bayes.predict_proba(X_vl_eng)[:, 1]
y_by_test_pred  = xgb_bayes.predict(X_te_eng)
y_by_test_proba = xgb_bayes.predict_proba(X_te_eng)[:, 1]

bayes_metrics = {
    'Accuracy'         : accuracy_score(y_test, y_by_test_pred),
    'Balanced Accuracy': balanced_accuracy_score(y_test, y_by_test_pred),
    'F1-Score'         : f1_score(y_test, y_by_test_pred),
    'ROC-AUC'          : roc_auc_score(y_test, y_by_test_proba),
}
bayes_val_metrics = {
    'Accuracy'         : accuracy_score(y_val, y_by_val_pred),
    'Balanced Accuracy': balanced_accuracy_score(y_val, y_by_val_pred),
    'F1-Score'         : f1_score(y_val, y_by_val_pred),
    'ROC-AUC'          : roc_auc_score(y_val, y_by_val_proba),
}

print(f'\n{"Metric":<22} {"Validation":>12} {"Test":>12}')
print('-'*48)
for k in bayes_metrics:
    print(f'{k:<22} {bayes_val_metrics[k]:>12.4f} {bayes_metrics[k]:>12.4f}')

print(f'\nTest Classification Report (Bayesian Opt):')
print(classification_report(y_test, y_by_test_pred, target_names=['No (0)', 'Yes (1)']))

In [ ]:
# ─────────────────────────────────────────────────────────────
# STEP 7 — PICK BEST MODEL + THRESHOLD TUNING
# ─────────────────────────────────────────────────────────────

all_search_results = {
    'Manual Tuned' : (tuned_val_metrics,  y_tu_val_proba,  y_tu_test_proba),
    'GridSearch'   : (gs_val_metrics,     y_gs_val_proba,  y_gs_test_proba),
    'RandomSearch' : (rs_val_metrics,     y_rs_val_proba,  y_rs_test_proba),
    'BayesOpt'     : (bayes_val_metrics,  y_by_val_proba,  y_by_test_proba),
}

# Pick best by val Balanced Accuracy
best_search_name = max(
    all_search_results,
    key=lambda k: all_search_results[k][0]['Balanced Accuracy']
)
_, best_val_proba, best_test_proba = all_search_results[best_search_name]

print(f'\n✅ Best search method: {best_search_name}')

# Print comparison table
print(f'\n{"Method":<16} {"Bal.Acc":>10} {"F1":>10} {"AUC":>10}')
print('-'*48)
for name, (m, _, _) in all_search_results.items():
    marker = ' ← best' if name == best_search_name else ''
    print(f'{name:<16} {m["Balanced Accuracy"]:>10.4f} {m["F1-Score"]:>10.4f} {m["ROC-AUC"]:>10.4f}{marker}')

# Now continue with threshold tuning using best_val_proba and best_test_proba
# (replace y_tu_val_proba and y_tu_test_proba in the existing Step 7b code)
y_tu_val_proba  = best_val_proba
y_tu_test_proba = best_test_proba

In [ ]:
# ─────────────────────────────────────────────────────────────
# STEP 8 — THRESHOLD TUNING FOR BALANCED ACCURACY
# ─────────────────────────────────────────────────────────────
# Default threshold = 0.5 optimises for accuracy on balanced data.
# For imbalanced data (~11% positive), a lower threshold catches
# more true positives (subscribers) at the cost of more false alarms.
# We search the val set for the threshold that maximises
# Balanced Accuracy, then apply it once to the test set.

from sklearn.metrics import balanced_accuracy_score
import numpy as np

print('\n' + '='*60)
print('  STEP 8 — THRESHOLD TUNING (Balanced Accuracy)')
print('='*60)

# Search thresholds on VALIDATION set only
thresholds    = np.arange(0.10, 0.60, 0.01)
best_thresh   = 0.5
best_bal_acc  = 0.0

thresh_results = []
for t in thresholds:
    y_val_t    = (y_tu_val_proba >= t).astype(int)
    bal_acc_t  = balanced_accuracy_score(y_val, y_val_t)
    thresh_results.append((t, bal_acc_t))
    if bal_acc_t > best_bal_acc:
        best_bal_acc = bal_acc_t
        best_thresh  = t

print(f'  Default threshold (0.50) → Val Bal.Acc: '
      f'{balanced_accuracy_score(y_val, (y_tu_val_proba >= 0.50).astype(int)):.4f}')
print(f'  Best threshold   ({best_thresh:.2f}) → Val Bal.Acc: {best_bal_acc:.4f}')

# Plot threshold vs Balanced Accuracy
thresh_arr   = [r[0] for r in thresh_results]
bal_acc_arr  = [r[1] for r in thresh_results]

plt.figure(figsize=(10, 5))
plt.plot(thresh_arr, bal_acc_arr, color='#4ECDC4', lw=2.5)
plt.axvline(x=best_thresh, color='#FF6B6B', linestyle='--', lw=2,
            label=f'Best threshold = {best_thresh:.2f}')
plt.axvline(x=0.50, color='#AAAAAA', linestyle='--', lw=1.5,
            label='Default threshold = 0.50')
plt.title('Threshold vs Balanced Accuracy (Validation Set)', fontweight='bold')
plt.xlabel('Threshold'); plt.ylabel('Balanced Accuracy')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# ── Apply best threshold to TEST set (only once) ─────────────
y_tu_test_pred_tuned  = (y_tu_test_proba >= best_thresh).astype(int)

tuned_thresh_metrics = {
    'Accuracy'         : accuracy_score(y_test,  y_tu_test_pred_tuned),
    'Balanced Accuracy': balanced_accuracy_score(y_test, y_tu_test_pred_tuned),
    'F1-Score'         : f1_score(y_test,  y_tu_test_pred_tuned),
    'ROC-AUC'          : roc_auc_score(y_test, y_tu_test_proba),  # unchanged
}

# ── FIX 2: Look up test metrics of the actual best model ─────
# Previously this was hardcoded to tuned_metrics (Manual Tuned),
# which is wrong if GridSearch / RandomSearch / BayesOpt won Step 7.
all_test_metrics = {
    'Manual Tuned' : tuned_metrics,
    'GridSearch'   : gs_metrics,
    'RandomSearch' : rs_metrics,
    'BayesOpt'     : bayes_metrics,
}
best_model_test_metrics = all_test_metrics[best_search_name]

# ── Delta table: best model (default 0.50) vs tuned threshold ─
print(f'\n  TUNED XGBoost + Threshold ({best_thresh:.2f})')
print(f'  Best model from Step 7   : {best_search_name}')
print('=' * 62)
print(f'{"Metric":<22} {"Default (0.50)":>15} {"Tuned Thresh":>14} {"Delta":>10}')
print('-'*62)
for k in best_model_test_metrics:
    delta = tuned_thresh_metrics[k] - best_model_test_metrics[k]
    arrow = '▲' if delta > 0 else ('▼' if delta < 0 else '–')
    print(f'{k:<22} {best_model_test_metrics[k]:>15.4f} '
          f'{tuned_thresh_metrics[k]:>14.4f} '
          f'{arrow}{abs(delta):>8.4f}')

print(f'\nTest Classification Report (threshold = {best_thresh:.2f}):')
print(classification_report(y_test, y_tu_test_pred_tuned,
                            target_names=['No (0)', 'Yes (1)']))

# Confusion matrix
cm_thresh = confusion_matrix(y_test, y_tu_test_pred_tuned)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cm_thresh, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Pred No', 'Pred Yes'],
            yticklabels=['Act No',  'Act Yes'])
axes[0].set_title(f'Confusion Matrix — Tuned XGBoost '
                  f'(threshold={best_thresh:.2f})', fontweight='bold')

fpr_th, tpr_th, _ = roc_curve(y_test, y_tu_test_proba)
axes[1].plot(fpr_th, tpr_th, color='#4ECDC4', lw=2.5,
             label=f'Tuned XGBoost (AUC={tuned_thresh_metrics["ROC-AUC"]:.3f})')
axes[1].plot([0,1],[0,1],'--', color='grey', label='Random')
axes[1].set_title('ROC Curve — Tuned XGBoost', fontweight='bold')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


print(f'\n✅ Threshold tuning complete.')

In [ ]:
# ─────────────────────────────────────────────────────────────
# STEP 9 — BASELINE vs TUNED XGBOOST COMPARISON
# ─────────────────────────────────────────────────────────────
# Both models use the same best resampling technique.
# This comparison isolates the effect of hyperparameter tuning.

print('\n' + '='*60)
print(f'  STEP 9 — BASELINE vs TUNED XGBoost ({best_technique})')
print('='*60)
print('  Baseline XGBoost : (no resampling, no tuning)')
print(f'  Tuned XGBoost    : ({best_technique} + hyperparameter tuning)')
print(f'{"Metric":<22} {"Baseline XGB":>13} {"Tuned XGB":>12} {"Delta":>10}')
print('-'*60)
for k in m_no_resample:
    delta = tuned_metrics[k] - m_no_resample[k]
    arrow = '▲' if delta > 0 else ('▼' if delta < 0 else '–')
    print(f'{k:<22} {m_no_resample[k]:>13.4f} {tuned_metrics[k]:>12.4f} '
          f'{arrow}{abs(delta):>8.4f}')

# ROC overlay
fpr_bl, tpr_bl, _ = roc_curve(y_test, model_nores.predict_proba(X_te_eng)[:, 1])
plt.figure(figsize=(8, 6))
plt.plot(fpr_bl, tpr_bl, color='#FF6B6B', lw=2,
         label=f'Baseline XGBoost (AUC={m_no_resample["ROC-AUC"]:.3f})')
plt.plot(fpr_tu, tpr_tu, color='#4ECDC4', lw=2,
         label=f'Tuned XGBoost    (AUC={tuned_metrics["ROC-AUC"]:.3f})')
plt.plot([0,1],[0,1],'--', color='grey', label='Random')
plt.title(f'ROC — Baseline vs Tuned XGBoost ({best_technique})', fontweight='bold')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# Confusion matrix side by side
cm_bl_step4 = confusion_matrix(y_test, model_nores.predict(X_te_eng))
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, cm, label in [(axes[0], cm_bl_step4, 'Baseline XGBoost (Step 4)'),
                       (axes[1], cm_tu,        'Tuned XGBoost (Step 7)')]:
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Pred No', 'Pred Yes'],
                yticklabels=['Act No',  'Act Yes'])
    ax.set_title(f'{label}', fontweight='bold')
fig.suptitle('Confusion Matrix Comparison — Baseline vs Tuned XGBoost',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print('\n' + '='*60)
print('  FINAL SUMMARY')
print('='*60)
print(f'  Best imbalance technique  : {best_technique}')
print(f'  Baseline XGBoost — Bal.Acc: {m_no_resample["Balanced Accuracy"]:.4f} | '
      f'F1: {m_no_resample["F1-Score"]:.4f} | AUC: {m_no_resample["ROC-AUC"]:.4f}')
print(f'  Tuned XGBoost    — Bal.Acc: {tuned_metrics["Balanced Accuracy"]:.4f} | '
      f'F1: {tuned_metrics["F1-Score"]:.4f} | AUC: {tuned_metrics["ROC-AUC"]:.4f}')
improvement = tuned_metrics["ROC-AUC"] - m_no_resample["ROC-AUC"]
direction   = "improvement" if improvement >= 0 else "degradation"
print(f'  Tuning {direction}          : {improvement:+.4f} AUC')
print('='*60)

In [ ]:
# milestone 4: Model Optimization


In [ ]:
# milestone 5: Final Evaluation 
